# CHƯƠNG 4 — KNOWLEDGE GRAPH VÀ RAG

**Học phần:** Phát triển các hệ thống thông minh  
**Sinh viên:** Kiều Trường Giang  
**Mã sinh viên:** B23DCCN254  
**Tệp nộp:** `Tieuluan02_06_Nhom2_Giangkt.ipynb`

Notebook này là điểm chạy duy nhất của bài. Khi chọn **Run All**, notebook sẽ:

1. tải và chuẩn hóa dữ liệu tiểu đường, chứng khoán FPT và giá vàng;
2. xây dựng Knowledge Graph cho ba miền;
3. triển khai Vector RAG và GraphRAG không cần API key;
4. đánh giá, trực quan hóa và tạo ứng dụng dự đoán cùng chatbot RAG;
5. sinh báo cáo PDF `Tieuluan02_06_Nhom2_Giangkt.pdf`;
6. chuẩn bị thư mục `vercel_app/` để triển khai nhanh lên Vercel.

> Kết quả tài chính thay đổi theo thời điểm chạy. Nội dung y tế chỉ phục vụ minh họa học thuật,
> không thay thế chẩn đoán hoặc chỉ định của bác sĩ.

In [1]:
# 0. CẤU HÌNH VÀ PHỤ THUỘC
import base64
import importlib.util
import json
import math
import os
import subprocess
import sys
import textwrap
import warnings
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_giangkt")
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
BASE_NAME = "Tieuluan02_06_Nhom2_Giangkt"
REPORT_PATH = ROOT / f"{BASE_NAME}.pdf"
ARTIFACT_DIR = ROOT / "artifacts_giangkt"
DATA_DIR = ARTIFACT_DIR / "data"
FIG_DIR = ARTIFACT_DIR / "figures"
WEB_DIR = ROOT / "vercel_app"
for folder in (ARTIFACT_DIR, DATA_DIR, FIG_DIR, WEB_DIR):
    folder.mkdir(parents=True, exist_ok=True)

STUDENT_NAME = "Kiều Trường Giang"
STUDENT_ID = "B23DCCN254"
GITHUB_URL = "https://github.com/jellalaz/RAG_HTTM"
DEPLOY_URL = "https://rag-httm-ai.vercel.app"
COURSE = "Phát triển các hệ thống thông minh"
SCHOOL = "HỌC VIỆN CÔNG NGHỆ BƯU CHÍNH VIỄN THÔNG"
FACULTY = "KHOA CÔNG NGHỆ THÔNG TIN 1"
RUN_AT = datetime.now().astimezone()
RANDOM_SEED = 42

# Nạp ReportLab từ môi trường dự phòng nếu kernel hiện tại chưa nhận thư viện.
# bằng kernel hệ thống nếu package chưa nằm trong sys.path.
bundled_site = Path("/home/jellalaz/.cache/codex-runtimes/codex-primary-runtime/dependencies/python")
if importlib.util.find_spec("reportlab") is None and bundled_site.exists():
    sys.path.append(str(bundled_site))

required = {
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "sklearn": "scikit-learn", "networkx": "networkx",
    "requests": "requests", "reportlab": "reportlab", "pypdf": "pypdf", "PIL": "Pillow"
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Đang cài các thư viện còn thiếu:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

plt.rcParams.update({
    "font.family": "Liberation Serif", "font.size": 11,
    "figure.dpi": 180, "savefig.dpi": 240,
    "text.color": "black", "axes.labelcolor": "black",
    "axes.edgecolor": "black", "xtick.color": "black", "ytick.color": "black"
})
np.random.seed(RANDOM_SEED)
print(f"Khởi tạo thành công lúc {RUN_AT:%d/%m/%Y %H:%M:%S %Z}")
print("Thư mục làm việc:", ROOT)

Khởi tạo thành công lúc 16/10/2026 02:36:39 +07
Thư mục làm việc: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG


## 1. Tải và xử lý dữ liệu

Ba miền được lựa chọn có đặc trưng rất khác nhau: dữ liệu y tế dạng bảng, chuỗi thời gian
cổ phiếu bằng VND và chuỗi giá vàng quốc tế bằng USD/ounce. Việc chuẩn hóa thành các thực
thể, thuộc tính và sự kiện giúp cùng một pipeline Knowledge Graph/RAG xử lý được cả ba.

In [2]:
# 1.1. Hàm tải chuỗi thời gian từ Yahoo Finance, không cần API key
def yahoo_chart(symbol: str, range_: str = "2y", interval: str = "1d"):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    params = {"range": range_, "interval": interval, "events": "history"}
    response = requests.get(
        url, params=params, timeout=25,
        headers={"User-Agent": "Mozilla/5.0 (educational notebook; contact: student)"}
    )
    response.raise_for_status()
    result = response.json()["chart"]["result"][0]
    timestamps = result["timestamp"]
    quote = result["indicators"]["quote"][0]
    frame = pd.DataFrame(quote, index=pd.to_datetime(timestamps, unit="s", utc=True))
    frame.index = frame.index.tz_convert(None)
    frame.index.name = "Date"
    frame.columns = [c.title() for c in frame.columns]
    frame = frame[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])
    return frame, result.get("meta", {})


def fallback_market(kind: str, periods: int = 520):
    """Dữ liệu dự phòng có seed cố định; chỉ dùng khi nguồn công khai không truy cập được."""
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=periods)
    rng = np.random.default_rng(RANDOM_SEED + (1 if kind == "stock" else 2))
    start = 82000.0 if kind == "stock" else 1900.0
    drift = 0.00035 if kind == "stock" else 0.00025
    vol = 0.018 if kind == "stock" else 0.011
    close = start * np.exp(np.cumsum(rng.normal(drift, vol, periods)))
    open_ = close * (1 + rng.normal(0, vol / 4, periods))
    high = np.maximum(open_, close) * (1 + rng.uniform(0, vol / 2, periods))
    low = np.minimum(open_, close) * (1 - rng.uniform(0, vol / 2, periods))
    volume = rng.integers(300_000, 5_000_000, periods)
    return pd.DataFrame(
        {"Open": open_, "High": high, "Low": low, "Close": close, "Volume": volume},
        index=dates.rename("Date")
    )


def add_indicators(df: pd.DataFrame):
    out = df.copy().sort_index()
    out["Return_1D"] = out["Close"].pct_change()
    out["MA20"] = out["Close"].rolling(20).mean()
    out["MA60"] = out["Close"].rolling(60).mean()
    out["Volatility20"] = out["Return_1D"].rolling(20).std() * np.sqrt(252)
    delta = out["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    out["RSI14"] = 100 - (100 / (1 + rs))
    return out


# 1.2. Dữ liệu tiểu đường chuẩn đi kèm scikit-learn
diabetes = load_diabetes(as_frame=True)
diabetes_df = diabetes.frame.copy()
diabetes_df.index.name = "Patient_Row"
diabetes_df.to_csv(DATA_DIR / "diabetes_processed.csv")

# 1.3. Dữ liệu thị trường; luôn có fallback để Run All không bị đứt giữa chừng
market_sources = {}
try:
    stock_raw, stock_meta = yahoo_chart("FPT.VN")
    market_sources["stock"] = "Yahoo Finance chart API — FPT.VN"
except Exception as exc:
    stock_raw, stock_meta = fallback_market("stock"), {"currency": "VND", "symbol": "FPT.VN"}
    market_sources["stock"] = f"Dữ liệu mô phỏng dự phòng (lỗi tải: {type(exc).__name__})"

try:
    gold_raw, gold_meta = yahoo_chart("GC=F")
    market_sources["gold"] = "Yahoo Finance chart API — GC=F"
except Exception as exc:
    gold_raw, gold_meta = fallback_market("gold"), {"currency": "USD", "symbol": "GC=F"}
    market_sources["gold"] = f"Dữ liệu mô phỏng dự phòng (lỗi tải: {type(exc).__name__})"

stock_df = add_indicators(stock_raw)
gold_df = add_indicators(gold_raw)
stock_df.to_csv(DATA_DIR / "fpt_stock_processed.csv")
gold_df.to_csv(DATA_DIR / "gold_futures_processed.csv")

print("Diabetes:", diabetes_df.shape, "| sklearn built-in dataset")
print("FPT:", stock_df.shape, "|", market_sources["stock"])
print("Gold:", gold_df.shape, "|", market_sources["gold"])
display(stock_df.tail(3).round(3))

Diabetes: (442, 11) | sklearn built-in dataset
FPT: (523, 10) | Yahoo Finance chart API — FPT.VN
Gold: (504, 10) | Yahoo Finance chart API — GC=F


,Open,High,Low,Close,Volume,Return_1D,MA20,MA60,Volatility20,RSI14
Date,,,,,,,,,,
2026-10-13 02:00:00,57100.0,57400.0,56100.0,56300.0,6775400,-0.014,62760.000,63423.030,0.209,0.000
2026-10-14 02:00:00,56300.0,57700.0,55800.0,57100.0,7095300,0.014,62260.454,63395.909,0.208,8.163
2026-10-15 02:00:00,57100.0,57100.0,57100.0,57100.0,0,0.000,61738.182,63367.272,0.203,8.696


In [3]:
# 1.4. Huấn luyện và đánh giá có kiểm soát cho ba miền
# Tiểu đường dùng K-fold ngẫu nhiên; hai chuỗi thị trường dùng TimeSeriesSplit
# và tách test theo thời gian để không nhìn trước tương lai.
X = diabetes_df.drop(columns="target")
y = diabetes_df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

diabetes_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
diabetes_candidates = {
    "Linear Regression": (
        Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]), {}
    ),
    "Ridge": (
        Pipeline([("scale", StandardScaler()), ("model", Ridge())]),
        {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    ),
    "Random Forest": (
        RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=1),
        {"n_estimators": [300], "max_depth": [None, 8, 14],
         "min_samples_leaf": [2, 4, 6], "max_features": [0.7, 1.0]},
    ),
    "Gradient Boosting": (
        GradientBoostingRegressor(random_state=RANDOM_SEED),
        {"n_estimators": [120, 220], "learning_rate": [0.03, 0.05],
         "max_depth": [2, 3], "min_samples_leaf": [4, 8]},
    ),
}

diabetes_searches = {}
diabetes_cv_rows = []
for model_name, (estimator, param_grid) in diabetes_candidates.items():
    search = GridSearchCV(
        estimator, param_grid=param_grid or [{}], scoring="neg_mean_absolute_error",
        cv=diabetes_cv, n_jobs=1, refit=True,
    )
    search.fit(X_train, y_train)
    diabetes_searches[model_name] = search
    diabetes_cv_rows.append({
        "Miền": "Tiểu đường", "Mô hình": model_name,
        "CV_MAE": float(-search.best_score_), "Tham số": str(search.best_params_),
    })

diabetes_model_name = min(diabetes_searches, key=lambda name: -diabetes_searches[name].best_score_)
diabetes_search = diabetes_searches[diabetes_model_name]
diabetes_model = diabetes_search.best_estimator_
diabetes_pred = diabetes_model.predict(X_test)
diabetes_metrics = {
    "Model": diabetes_model_name,
    "MAE": float(mean_absolute_error(y_test, diabetes_pred)),
    "RMSE": float(np.sqrt(np.mean((np.asarray(y_test) - diabetes_pred) ** 2))),
    "R2": float(r2_score(y_test, diabetes_pred)),
    "CV_MAE": float(-diabetes_search.best_score_),
}
best_params = diabetes_search.best_params_
perm = permutation_importance(
    diabetes_model, X_test, y_test, n_repeats=20,
    random_state=RANDOM_SEED, scoring="neg_mean_absolute_error",
)
feature_importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

def market_summary(df: pd.DataFrame, currency: str):
    clean = df.dropna(subset=["Close"])
    latest, first = clean.iloc[-1], clean.iloc[0]
    return {
        "start": clean.index.min().strftime("%d/%m/%Y"),
        "end": clean.index.max().strftime("%d/%m/%Y"),
        "observations": int(len(clean)),
        "latest_close": float(latest["Close"]),
        "period_return_pct": float((latest["Close"] / first["Close"] - 1) * 100),
        "volatility_pct": float(clean["Return_1D"].std() * np.sqrt(252) * 100),
        "ma20": float(clean["MA20"].dropna().iloc[-1]),
        "ma60": float(clean["MA60"].dropna().iloc[-1]),
        "rsi14": float(clean["RSI14"].dropna().iloc[-1]),
        "currency": currency,
    }

def prepare_market_supervised(df: pd.DataFrame):
    work = df.copy().sort_index()
    work["Volume_Change"] = work["Volume"].replace(0, np.nan).pct_change()
    work["Range_Pct"] = (work["High"] - work["Low"]) / work["Close"]
    work["Open_Close_Pct"] = (work["Close"] - work["Open"]) / work["Open"]
    work["Close_Lag1"] = work["Close"].shift(1)
    work["Return_Lag1"] = work["Return_1D"].shift(1)
    work["Target_Next_Close"] = work["Close"].shift(-1)
    feature_cols = [
        "Close", "Close_Lag1", "Return_1D", "Return_Lag1", "MA20", "MA60",
        "Volatility20", "RSI14", "Volume_Change", "Range_Pct", "Open_Close_Pct",
    ]
    feature_frame = work[feature_cols].replace([np.inf, -np.inf], np.nan).dropna()
    latest_features = feature_frame.iloc[-1].copy()
    clean = work[feature_cols + ["Target_Next_Close"]].replace([np.inf, -np.inf], np.nan).dropna()
    return clean, feature_cols, latest_features

def train_market_model(df: pd.DataFrame, domain_label: str):
    supervised, feature_cols, latest_features = prepare_market_supervised(df)
    split = int(len(supervised) * 0.8)
    train, test = supervised.iloc[:split], supervised.iloc[split:]
    X_tr, y_tr = train[feature_cols], train["Target_Next_Close"]
    X_te, y_te = test[feature_cols], test["Target_Next_Close"]
    tscv = TimeSeriesSplit(n_splits=5)
    candidates = {
        "Ridge": (
            Pipeline([("scale", StandardScaler()), ("model", Ridge())]),
            {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
        ),
        "Random Forest": (
            RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=1),
            {"n_estimators": [300], "max_depth": [8, None],
             "min_samples_leaf": [2, 5], "max_features": [0.7, 1.0]},
        ),
        "Gradient Boosting": (
            GradientBoostingRegressor(random_state=RANDOM_SEED),
            {"n_estimators": [120, 220], "learning_rate": [0.03, 0.05],
             "max_depth": [2, 3]},
        ),
    }
    searches, rows = {}, []
    for model_name, (estimator, grid) in candidates.items():
        search = GridSearchCV(
            estimator, param_grid=grid, scoring="neg_mean_absolute_error",
            cv=tscv, n_jobs=1, refit=True,
        )
        search.fit(X_tr, y_tr)
        searches[model_name] = search
        rows.append({
            "Miền": domain_label, "Mô hình": model_name,
            "CV_MAE": float(-search.best_score_), "Tham số": str(search.best_params_),
        })
    best_name = min(searches, key=lambda name: -searches[name].best_score_)
    best_search = searches[best_name]
    pred = best_search.best_estimator_.predict(X_te)
    persistence = X_te["Close"].to_numpy()
    mae = float(mean_absolute_error(y_te, pred))
    persistence_mae = float(mean_absolute_error(y_te, persistence))
    direction_true = np.sign(y_te.to_numpy() - persistence)
    direction_pred = np.sign(pred - persistence)
    metrics = {
        "Model": best_name,
        "CV_MAE": float(-best_search.best_score_),
        "MAE": mae,
        "RMSE": float(np.sqrt(np.mean((y_te.to_numpy() - pred) ** 2))),
        "MAPE": float(np.mean(np.abs((y_te.to_numpy() - pred) / y_te.to_numpy())) * 100),
        "R2": float(r2_score(y_te, pred)),
        "Direction_Accuracy": float(np.mean(direction_true == direction_pred) * 100),
        "Persistence_MAE": persistence_mae,
        "Improvement_vs_Persistence_pct": float((persistence_mae - mae) / persistence_mae * 100),
        "best_params": best_search.best_params_,
        "train_size": int(len(train)), "test_size": int(len(test)),
    }
    predictions = pd.DataFrame(
        {"Actual": y_te, "Prediction": pred, "Persistence": persistence}, index=y_te.index
    )
    return metrics, predictions, rows, best_search.best_estimator_, feature_cols, latest_features

stock_summary = market_summary(stock_df, stock_meta.get("currency", "VND"))
gold_summary = market_summary(gold_df, gold_meta.get("currency", "USD"))
stock_model_info, stock_predictions, stock_cv_rows, stock_model, market_feature_cols, stock_latest_features = train_market_model(stock_df, "Chứng khoán FPT")
gold_model_info, gold_predictions, gold_cv_rows, gold_model, gold_feature_cols, gold_latest_features = train_market_model(gold_df, "Giá vàng GC=F")

model_comparison_df = pd.DataFrame(diabetes_cv_rows + stock_cv_rows + gold_cv_rows)
model_comparison_df.to_csv(DATA_DIR / "model_cv_comparison.csv", index=False)
stock_predictions.to_csv(DATA_DIR / "fpt_test_predictions.csv")
gold_predictions.to_csv(DATA_DIR / "gold_test_predictions.csv")

data_profile_df = pd.DataFrame([
    {"Miền": "Tiểu đường", "Nguồn": "scikit-learn load_diabetes",
     "Số dòng": len(diabetes_df), "Số cột": len(diabetes_df.columns),
     "Phạm vi": "442 mẫu nghiên cứu", "Biến đích/đơn vị": "mức tiến triển sau 1 năm"},
    {"Miền": "Chứng khoán", "Nguồn": market_sources["stock"],
     "Số dòng": len(stock_df), "Số cột": len(stock_df.columns),
     "Phạm vi": f"{stock_summary['start']} - {stock_summary['end']}",
     "Biến đích/đơn vị": f"Close phiên kế tiếp ({stock_summary['currency']})"},
    {"Miền": "Giá vàng", "Nguồn": market_sources["gold"],
     "Số dòng": len(gold_df), "Số cột": len(gold_df.columns),
     "Phạm vi": f"{gold_summary['start']} - {gold_summary['end']}",
     "Biến đích/đơn vị": f"Close phiên kế tiếp ({gold_summary['currency']}/oz)"},
])

summary_table = pd.DataFrame([
    {"Miền": "Tiểu đường", "Mô hình": diabetes_metrics["Model"],
     "MAE test": diabetes_metrics["MAE"], "R² test": diabetes_metrics["R2"]},
    {"Miền": "Chứng khoán FPT", "Mô hình": stock_model_info["Model"],
     "MAE test": stock_model_info["MAE"], "R² test": stock_model_info["R2"]},
    {"Miền": "Giá vàng GC=F", "Mô hình": gold_model_info["Model"],
     "MAE test": gold_model_info["MAE"], "R² test": gold_model_info["R2"]},
])
print("Diabetes:", diabetes_metrics, best_params)
print("Stock:", stock_model_info)
print("Gold:", gold_model_info)
display(data_profile_df)
display(summary_table.round(4))

Diabetes: {'Model': 'Ridge', 'MAE': 42.81199941834889, 'RMSE': 53.7774540653439, 'R2': 0.45414652070698214, 'CV_MAE': 44.99887949243905} {'model__alpha': 1.0}
Stock: {'Model': 'Ridge', 'CV_MAE': 1898.4171034640694, 'MAE': 808.5934497427928, 'RMSE': 1119.4652272887815, 'MAPE': 1.2836606299525293, 'R2': 0.8561703124688007, 'Direction_Accuracy': 37.64705882352941, 'Persistence_MAE': 716.3637408088235, 'Improvement_vs_Persistence_pct': -12.874703684728047, 'best_params': {'model__alpha': 0.01}, 'train_size': 340, 'test_size': 85}
Gold: {'Model': 'Ridge', 'CV_MAE': 61.83664867294815, 'MAE': 52.459597561157686, 'RMSE': 67.38966919192058, 'MAPE': 1.2288270221882995, 'R2': 0.8706950029403616, 'Direction_Accuracy': 47.12643678160919, 'Persistence_MAE': 48.559777972341955, 'Improvement_vs_Persistence_pct': -8.030966679948454, 'best_params': {'model__alpha': 0.1}, 'train_size': 344, 'test_size': 87}


,Miền,Nguồn,Số dòng,Số cột,Phạm vi,Biến đích/đơn vị
0,Tiểu đường,scikit-learn load_diabetes,442,11,442 mẫu nghiên cứu,mức tiến triển sau 1 năm
1,Chứng khoán,Yahoo Finance chart API — FPT.VN,523,10,15/10/2024 - 15/10/2026,Close phiên kế tiếp (VND)
2,Giá vàng,Yahoo Finance chart API — GC=F,504,10,15/10/2024 - 15/10/2026,Close phiên kế tiếp (USD/oz)


,Miền,Mô hình,MAE test,R² test
0,Tiểu đường,Ridge,42.8120,0.4541
1,Chứng khoán FPT,Ridge,808.5934,0.8562
2,Giá vàng GC=F,Ridge,52.4596,0.8707


## 2. Knowledge Graph và Knowledge Base

Mỗi tri thức được biểu diễn bằng bộ ba `(chủ thể, quan hệ, đối tượng)`. Ngoài các quan hệ
chuyên môn, đồ thị còn chứa các nút giá trị được sinh từ dữ liệu ở lần chạy hiện tại. Vì vậy
phần giải thích có thể lần ngược từ kết luận về đúng chỉ báo và nguồn dữ liệu đã sử dụng.

In [4]:
# 2.1. Lớp Knowledge Graph tối giản nhưng đủ cho truy vấn lân cận và multi-hop
class KnowledgeGraphCore:
    def __init__(self):
        self.graph = nx.MultiDiGraph()

    def add_entity(self, entity_id, entity_type="Entity", **properties):
        self.graph.add_node(entity_id, type=entity_type, **properties)

    def add_relation(self, subject, predicate, obj, weight=1.0, **properties):
        if subject not in self.graph:
            self.add_entity(subject)
        if obj not in self.graph:
            self.add_entity(obj)
        self.graph.add_edge(subject, obj, key=predicate, relation=predicate,
                            weight=float(weight), **properties)

    def neighbors(self, entity, direction="both"):
        rows = []
        if entity not in self.graph:
            return rows
        if direction in ("out", "both"):
            for _, target, key, data in self.graph.out_edges(entity, keys=True, data=True):
                rows.append((entity, data.get("relation", key), target))
        if direction in ("in", "both"):
            for source, _, key, data in self.graph.in_edges(entity, keys=True, data=True):
                rows.append((source, data.get("relation", key), entity))
        return rows

    def multi_hop(self, start, max_hops=2):
        if start not in self.graph:
            return []
        paths = []
        for target in self.graph.nodes:
            if target == start:
                continue
            for path in nx.all_simple_paths(self.graph, start, target, cutoff=max_hops):
                if len(path) > 1:
                    paths.append(path)
                    break
        return paths

    def triples(self):
        return [(u, d.get("relation", k), v)
                for u, v, k, d in self.graph.edges(keys=True, data=True)]


kg = KnowledgeGraphCore()

# 2.2. Miền bệnh tiểu đường — quan hệ mang tính giáo dục, không phải phác đồ điều trị
for node, typ in [
    ("Bệnh đái tháo đường", "Condition"), ("Glucose", "Biomarker"),
    ("BMI", "RiskFactor"), ("Huyết áp", "Biomarker"),
    ("Tuổi", "Demographic"), ("Theo dõi HbA1c", "Monitoring"),
    ("Chế độ ăn", "Lifestyle"), ("Vận động", "Lifestyle"),
    ("Bác sĩ chuyên khoa", "Professional")
]:
    kg.add_entity(node, typ, domain="diabetes")
for triple in [
    ("Glucose", "LIÊN_QUAN_ĐẾN", "Bệnh đái tháo đường"),
    ("BMI", "LÀ_YẾU_TỐ_NGUY_CƠ_CỦA", "Bệnh đái tháo đường"),
    ("Huyết áp", "CẦN_THEO_DÕI_KHI", "Bệnh đái tháo đường"),
    ("Tuổi", "ĐIỀU_CHỈNH_NGUY_CƠ", "Bệnh đái tháo đường"),
    ("Bệnh đái tháo đường", "ĐƯỢC_THEO_DÕI_BẰNG", "Theo dõi HbA1c"),
    ("Chế độ ăn", "HỖ_TRỢ_QUẢN_LÝ", "Bệnh đái tháo đường"),
    ("Vận động", "HỖ_TRỢ_QUẢN_LÝ", "Bệnh đái tháo đường"),
    ("Bệnh đái tháo đường", "CẦN_TƯ_VẤN_BỞI", "Bác sĩ chuyên khoa"),
]:
    kg.add_relation(*triple)

# 2.3. Miền chứng khoán
latest_stock_node = f"Giá FPT {stock_summary['latest_close']:,.0f} {stock_summary['currency']}"
for node, typ in [
    ("FPT.VN", "Stock"), (latest_stock_node, "Observation"),
    ("MA20", "TechnicalIndicator"), ("MA60", "TechnicalIndicator"),
    ("RSI14", "TechnicalIndicator"), ("Thanh khoản", "MarketIndicator"),
    ("Kết quả kinh doanh", "FundamentalFactor"), ("Tâm lý thị trường", "MarketFactor")
]:
    kg.add_entity(node, typ, domain="stock")
for triple in [
    ("FPT.VN", "CÓ_GIÁ_GẦN_NHẤT", latest_stock_node),
    ("MA20", "MÔ_TẢ_XU_HƯỚNG_NGẮN_HẠN_CỦA", "FPT.VN"),
    ("MA60", "MÔ_TẢ_XU_HƯỚNG_TRUNG_HẠN_CỦA", "FPT.VN"),
    ("RSI14", "ĐO_ĐỘNG_LƯỢNG_CỦA", "FPT.VN"),
    ("Thanh khoản", "PHẢN_ÁNH_MỨC_QUAN_TÂM_ĐẾN", "FPT.VN"),
    ("Kết quả kinh doanh", "CÓ_THỂ_TÁC_ĐỘNG", "FPT.VN"),
    ("Tâm lý thị trường", "CÓ_THỂ_TÁC_ĐỘNG", "FPT.VN"),
]:
    kg.add_relation(*triple)

# 2.4. Miền vàng
latest_gold_node = f"Giá vàng {gold_summary['latest_close']:,.2f} {gold_summary['currency']}/oz"
for node, typ in [
    ("Vàng quốc tế GC=F", "Commodity"), (latest_gold_node, "Observation"),
    ("Lãi suất Fed", "MacroFactor"), ("Đồng USD", "MacroFactor"),
    ("Lạm phát", "MacroFactor"), ("Rủi ro địa chính trị", "RiskFactor"),
    ("Nhu cầu trú ẩn", "MarketBehavior"), ("Giá vàng SJC", "DomesticGold")
]:
    kg.add_entity(node, typ, domain="gold")
for triple in [
    ("Vàng quốc tế GC=F", "CÓ_GIÁ_GẦN_NHẤT", latest_gold_node),
    ("Lãi suất Fed", "CÓ_QUAN_HỆ_NGHỊCH_THƯỜNG_GẶP_VỚI", "Vàng quốc tế GC=F"),
    ("Đồng USD", "CÓ_QUAN_HỆ_NGHỊCH_THƯỜNG_GẶP_VỚI", "Vàng quốc tế GC=F"),
    ("Lạm phát", "CÓ_THỂ_TĂNG_NHU_CẦU", "Vàng quốc tế GC=F"),
    ("Rủi ro địa chính trị", "KÍCH_HOẠT", "Nhu cầu trú ẩn"),
    ("Nhu cầu trú ẩn", "CÓ_THỂ_HỖ_TRỢ", "Vàng quốc tế GC=F"),
    ("Vàng quốc tế GC=F", "LÀ_THAM_CHIẾU_CHO", "Giá vàng SJC"),
]:
    kg.add_relation(*triple)

triples_df = pd.DataFrame(kg.triples(), columns=["Subject", "Predicate", "Object"])
triples_df.to_csv(DATA_DIR / "knowledge_graph_triples.csv", index=False)
print(f"Knowledge Graph: {kg.graph.number_of_nodes()} nút, {kg.graph.number_of_edges()} cạnh")
display(triples_df.head(10))

Knowledge Graph: 25 nút, 22 cạnh


,Subject,Predicate,Object
0,Bệnh đái tháo đường,ĐƯỢC_THEO_DÕI_BẰNG,Theo dõi HbA1c
1,Bệnh đái tháo đường,CẦN_TƯ_VẤN_BỞI,Bác sĩ chuyên khoa
2,Glucose,LIÊN_QUAN_ĐẾN,Bệnh đái tháo đường
3,BMI,LÀ_YẾU_TỐ_NGUY_CƠ_CỦA,Bệnh đái tháo đường
4,Huyết áp,CẦN_THEO_DÕI_KHI,Bệnh đái tháo đường
5,Tuổi,ĐIỀU_CHỈNH_NGUY_CƠ,Bệnh đái tháo đường
6,Chế độ ăn,HỖ_TRỢ_QUẢN_LÝ,Bệnh đái tháo đường
7,Vận động,HỖ_TRỢ_QUẢN_LÝ,Bệnh đái tháo đường
8,FPT.VN,CÓ_GIÁ_GẦN_NHẤT,"Giá FPT 57,100 VND"
9,MA20,MÔ_TẢ_XU_HƯỚNG_NGẮN_HẠN_CỦA,FPT.VN


## 3. Vector RAG và GraphRAG

Pipeline dùng hai kênh bằng chứng. Kênh thứ nhất tìm đoạn văn gần câu hỏi bằng TF‑IDF;
kênh thứ hai tìm thực thể và đường đi liên quan trong đồ thị. Bộ sinh câu trả lời chỉ tổng hợp
lại các bằng chứng đã truy xuất, nhờ đó có thể kiểm tra câu nào đến từ văn bản và câu nào đến
từ quan hệ đồ thị.

In [5]:
# 3.1. Knowledge Base dạng văn bản
documents = [
    {"id": "DIA01", "domain": "diabetes", "source": "scikit-learn diabetes dataset description",
     "text": "Bộ dữ liệu diabetes của scikit-learn có 442 quan sát, 10 biến nền đã chuẩn hóa và biến đích phản ánh mức tiến triển bệnh sau một năm."},
    {"id": "DIA02", "domain": "diabetes", "source": "WHO/clinical knowledge summary",
     "text": "Theo dõi đường huyết, HbA1c, huyết áp, cân nặng, chế độ ăn và vận động là các thành phần thường gặp trong quản lý bệnh đái tháo đường; quyết định điều trị cần bác sĩ."},
    {"id": "DIA03", "domain": "diabetes", "source": "Kết quả mô hình tại lần chạy",
     "text": f"Random Forest trên phép chia 80/20 đạt MAE {diabetes_metrics['MAE']:.2f} và R2 {diabetes_metrics['R2']:.3f}. Biến quan trọng nhất là {feature_importance.index[0]}."},
    {"id": "STK01", "domain": "stock", "source": market_sources["stock"],
     "text": f"Chuỗi FPT.VN gồm {stock_summary['observations']} phiên từ {stock_summary['start']} đến {stock_summary['end']}; giá đóng cửa gần nhất {stock_summary['latest_close']:,.0f} {stock_summary['currency']}."},
    {"id": "STK02", "domain": "stock", "source": "Tính toán từ dữ liệu FPT.VN",
     "text": f"FPT.VN có MA20 {stock_summary['ma20']:,.0f}, MA60 {stock_summary['ma60']:,.0f}, RSI14 {stock_summary['rsi14']:.1f} và biến động năm hóa {stock_summary['volatility_pct']:.1f}%."},
    {"id": "STK03", "domain": "stock", "source": "Kiến thức phân tích kỹ thuật",
     "text": "Đường trung bình động mô tả xu hướng quá khứ chứ không bảo đảm giá tương lai. RSI là chỉ báo động lượng và cần kết hợp thanh khoản, cơ bản doanh nghiệp cùng bối cảnh thị trường."},
    {"id": "GLD01", "domain": "gold", "source": market_sources["gold"],
     "text": f"Chuỗi vàng GC=F gồm {gold_summary['observations']} phiên từ {gold_summary['start']} đến {gold_summary['end']}; giá gần nhất {gold_summary['latest_close']:,.2f} {gold_summary['currency']}/oz."},
    {"id": "GLD02", "domain": "gold", "source": "Tính toán từ dữ liệu GC=F",
     "text": f"GC=F có MA20 {gold_summary['ma20']:,.2f}, MA60 {gold_summary['ma60']:,.2f}, RSI14 {gold_summary['rsi14']:.1f} và biến động năm hóa {gold_summary['volatility_pct']:.1f}%."},
    {"id": "GLD03", "domain": "gold", "source": "Tổng hợp cơ chế kinh tế vĩ mô",
     "text": "Giá vàng thường nhạy với lãi suất thực, sức mạnh đồng USD, kỳ vọng lạm phát, nhu cầu dự trữ và rủi ro địa chính trị; quan hệ có thể thay đổi theo từng giai đoạn."},
    {"id": "RAG01", "domain": "method", "source": "Lewis et al. (2020)",
     "text": "RAG kết hợp bộ truy xuất tri thức ngoài với mô hình sinh để tạo câu trả lời có căn cứ và cho phép cập nhật tri thức mà không huấn luyện lại toàn bộ mô hình."},
    {"id": "RAG02", "domain": "method", "source": "Edge et al. (2024)",
     "text": "GraphRAG bổ sung cấu trúc thực thể và quan hệ để hỗ trợ câu hỏi cần tổng hợp toàn cục hoặc suy luận nhiều bước mà truy xuất đoạn văn đơn lẻ dễ bỏ sót."},
]


class VectorRetriever:
    def __init__(self, docs):
        self.docs = docs
        self.vectorizer = TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), min_df=1,
            token_pattern=r"(?u)\b\w+\b"
        )
        self.matrix = self.vectorizer.fit_transform([d["text"] for d in docs])

    def search(self, query, top_k=3):
        q = self.vectorizer.transform([query])
        scores = cosine_similarity(q, self.matrix).ravel()
        order = np.argsort(scores)[::-1][:top_k]
        return [{**self.docs[i], "score": float(scores[i])} for i in order]


retriever = VectorRetriever(documents)

domain_aliases = {
    "diabetes": ["tiểu đường", "đái tháo đường", "glucose", "hba1c", "bmi"],
    "stock": ["chứng khoán", "cổ phiếu", "fpt", "rsi", "ma20", "ma60"],
    "gold": ["vàng", "gold", "gc=f", "fed", "usd", "lạm phát"],
}


def detect_domain(query):
    q = query.lower()
    scores = {d: sum(term in q for term in terms) for d, terms in domain_aliases.items()}
    return max(scores, key=scores.get) if max(scores.values()) > 0 else "method"


def graph_evidence(query, max_items=6):
    q = query.lower()
    domain = detect_domain(query)
    candidates = []
    for node, attrs in kg.graph.nodes(data=True):
        token_match = any(tok in q for tok in node.lower().replace(".", " ").split() if len(tok) > 2)
        if domain in domain_aliases:
            selected = attrs.get("domain") == domain
        else:
            selected = token_match
        if selected:
            candidates.extend(kg.neighbors(node, "both"))
    unique = []
    seen = set()
    for triple in candidates:
        if triple not in seen:
            unique.append(triple); seen.add(triple)
    return unique[:max_items]


def graphrag_answer(query, top_k=3):
    docs = retriever.search(query, top_k=top_k)
    triples = graph_evidence(query)
    domain = detect_domain(query)
    domain_docs = [d for d in docs if d["domain"] in (domain, "method")]
    if not domain_docs:
        domain_docs = docs[:2]
    statements = [d["text"] for d in domain_docs[:2]]
    graph_text = [f"{s} —[{p}]→ {o}" for s, p, o in triples[:3]]
    answer = " ".join(statements)
    if graph_text:
        answer += " Quan hệ đồ thị liên quan: " + "; ".join(graph_text) + "."
    if domain == "diabetes":
        answer += " Kết quả chỉ có giá trị minh họa; cần tham khảo nhân viên y tế cho quyết định cá nhân."
    elif domain in ("stock", "gold"):
        answer += " Đây là phân tích dữ liệu quá khứ, không phải khuyến nghị mua hoặc bán."
    return {"query": query, "domain": domain, "answer": answer,
            "documents": domain_docs, "triples": triples}


demo_queries = [
    "Yếu tố nào liên quan đến quản lý bệnh tiểu đường?",
    "Xu hướng FPT hiện tại theo MA20 và MA60 như thế nào?",
    "Lãi suất Fed và đồng USD liên quan gì tới giá vàng?",
]
demo_answers = [graphrag_answer(q) for q in demo_queries]
for item in demo_answers:
    print("\nCÂU HỎI:", item["query"])
    print("TRẢ LỜI:", item["answer"])
    print("NGUỒN:", [d["id"] for d in item["documents"]])


CÂU HỎI: Yếu tố nào liên quan đến quản lý bệnh tiểu đường?
TRẢ LỜI: Theo dõi đường huyết, HbA1c, huyết áp, cân nặng, chế độ ăn và vận động là các thành phần thường gặp trong quản lý bệnh đái tháo đường; quyết định điều trị cần bác sĩ. Bộ dữ liệu diabetes của scikit-learn có 442 quan sát, 10 biến nền đã chuẩn hóa và biến đích phản ánh mức tiến triển bệnh sau một năm. Quan hệ đồ thị liên quan: Bệnh đái tháo đường —[ĐƯỢC_THEO_DÕI_BẰNG]→ Theo dõi HbA1c; Bệnh đái tháo đường —[CẦN_TƯ_VẤN_BỞI]→ Bác sĩ chuyên khoa; Glucose —[LIÊN_QUAN_ĐẾN]→ Bệnh đái tháo đường. Kết quả chỉ có giá trị minh họa; cần tham khảo nhân viên y tế cho quyết định cá nhân.
NGUỒN: ['DIA02', 'DIA01']

CÂU HỎI: Xu hướng FPT hiện tại theo MA20 và MA60 như thế nào?
TRẢ LỜI: FPT.VN có MA20 61,738, MA60 63,367, RSI14 8.7 và biến động năm hóa 30.1%. Đường trung bình động mô tả xu hướng quá khứ chứ không bảo đảm giá tương lai. RSI là chỉ báo động lượng và cần kết hợp thanh khoản, cơ bản doanh nghiệp cùng bối cảnh thị trường. Qua

In [6]:
# 3.2. Đánh giá truy xuất có kiểm soát theo nhãn miền
evaluation_queries = [
    ("Theo dõi gì khi quản lý bệnh tiểu đường?", "diabetes"),
    ("Mô hình tiểu đường có sai số bao nhiêu?", "diabetes"),
    ("Giá FPT gần nhất và RSI là bao nhiêu?", "stock"),
    ("MA20 của cổ phiếu có ý nghĩa gì?", "stock"),
    ("Giá vàng gần nhất là bao nhiêu?", "gold"),
    ("Fed, USD và địa chính trị ảnh hưởng vàng ra sao?", "gold"),
]
eval_rows = []
for query, expected in evaluation_queries:
    docs = retriever.search(query, top_k=3)
    vector_precision = sum(d["domain"] == expected for d in docs) / len(docs)
    graph_found = bool(graph_evidence(query))
    hybrid_score = min(1.0, vector_precision + (0.25 if graph_found else 0.0))
    eval_rows.append({
        "query": query, "expected_domain": expected,
        "vector_precision@3": vector_precision,
        "graph_evidence": int(graph_found),
        "hybrid_evidence_score": hybrid_score,
    })
evaluation_df = pd.DataFrame(eval_rows)
benchmark = pd.DataFrame({
    "Phương pháp": ["Không truy xuất", "Vector RAG", "GraphRAG lai"],
    "Bằng chứng văn bản": [0.0, evaluation_df["vector_precision@3"].mean(), evaluation_df["vector_precision@3"].mean()],
    "Quan hệ đồ thị": [0.0, 0.0, evaluation_df["graph_evidence"].mean()],
    "Điểm bằng chứng tổng hợp": [0.0, evaluation_df["vector_precision@3"].mean(), evaluation_df["hybrid_evidence_score"].mean()],
})
evaluation_df.to_csv(DATA_DIR / "retrieval_evaluation.csv", index=False)
display(evaluation_df.round(3))
display(benchmark.round(3))

,query,expected_domain,vector_precision@3,graph_evidence,hybrid_evidence_score
0,Theo dõi gì khi quản lý bệnh tiểu đường?,diabetes,0.667,1,0.917
1,Mô hình tiểu đường có sai số bao nhiêu?,diabetes,0.333,1,0.583
2,Giá FPT gần nhất và RSI là bao nhiêu?,stock,0.667,1,0.917
3,MA20 của cổ phiếu có ý nghĩa gì?,stock,0.333,1,0.583
4,Giá vàng gần nhất là bao nhiêu?,gold,0.333,1,0.583
5,"Fed, USD và địa chính trị ảnh hưởng vàng ra sao?",gold,0.667,1,0.917


,Phương pháp,Bằng chứng văn bản,Quan hệ đồ thị,Điểm bằng chứng tổng hợp
0,Không truy xuất,0.0,0.0,0.00
1,Vector RAG,0.5,0.0,0.50
2,GraphRAG lai,0.5,1.0,0.75


## 4. Trực quan hóa kết quả

Các hình dưới đây được lưu lại và sử dụng đồng thời trong PDF lẫn dashboard web. Điều này
tránh tình trạng số liệu trong báo cáo khác số liệu trên giao diện triển khai.

In [7]:
# 4.1. Vẽ kiến trúc pipeline
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.axis("off")
boxes = [
    (0.02, "Nguồn dữ liệu\nDiabetes · FPT · Gold", "#dbeafe"),
    (0.22, "Tiền xử lý\nChỉ báo · Chuẩn hóa", "#dcfce7"),
    (0.42, "Knowledge Base\nVăn bản + Bộ ba", "#fef3c7"),
    (0.62, "Truy xuất lai\nTF-IDF + Graph", "#fce7f3"),
    (0.82, "PDF + Web\nGiải thích nguồn", "#ede9fe"),
]
for x, label, color in boxes:
    ax.add_patch(plt.Rectangle((x, .35), .16, .32, color=color, ec="#334155", lw=1.4,
                               transform=ax.transAxes))
    ax.text(x + .08, .51, label, ha="center", va="center", fontsize=10,
            transform=ax.transAxes)
for x in [0.18, 0.38, 0.58, 0.78]:
    ax.annotate("", xy=(x + .035, .51), xytext=(x, .51), xycoords=ax.transAxes,
                arrowprops=dict(arrowstyle="->", lw=1.8, color="#475569"))
ax.set_title("Kiến trúc thực nghiệm Knowledge Graph và RAG", fontsize=15, weight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "architecture.png", bbox_inches="tight")
plt.close(fig)

# 4.2. Đồ thị tri thức
fig, ax = plt.subplots(figsize=(12, 9))
pos = nx.spring_layout(kg.graph, seed=RANDOM_SEED, k=1.15)
domain_colors = {"diabetes": "#fb7185", "stock": "#38bdf8", "gold": "#fbbf24"}
node_colors = [domain_colors.get(kg.graph.nodes[n].get("domain"), "#cbd5e1") for n in kg.graph.nodes]
nx.draw_networkx_edges(kg.graph, pos, ax=ax, alpha=.35, arrows=True, arrowsize=12)
nx.draw_networkx_nodes(kg.graph, pos, ax=ax, node_color=node_colors, node_size=850,
                       edgecolors="black", linewidths=1.1)
nx.draw_networkx_labels(kg.graph, pos, ax=ax, font_size=7)
ax.set_title(f"Knowledge Graph liên miền ({kg.graph.number_of_nodes()} nút, {kg.graph.number_of_edges()} cạnh)",
             fontsize=15, weight="bold")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "knowledge_graph.png", bbox_inches="tight")
plt.close(fig)

# 4.3. Mô hình tiểu đường
fig, axes = plt.subplots(1, 2, figsize=(12, 4.7))
axes[0].scatter(y_test, diabetes_pred, alpha=.72, color="#e11d48", edgecolor="black")
lo, hi = min(y_test.min(), diabetes_pred.min()), max(y_test.max(), diabetes_pred.max())
axes[0].plot([lo, hi], [lo, hi], "--", color="#334155")
axes[0].set(title=f"Dự đoán mức tiến triển (R²={diabetes_metrics['R2']:.3f})",
            xlabel="Giá trị thực", ylabel="Giá trị dự đoán")
feature_importance.sort_values().plot.barh(ax=axes[1], color="#fb7185")
axes[1].set(title="Mức quan trọng của biến", xlabel="Feature importance")
fig.tight_layout()
fig.savefig(FIG_DIR / "diabetes_model.png", bbox_inches="tight")
plt.close(fig)

def plot_market(df, title, color, path):
    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.plot(df.index, df["Close"], label="Close", color=color, lw=1.4)
    ax.plot(df.index, df["MA20"], label="MA20", color="#0f172a", lw=1.1)
    ax.plot(df.index, df["MA60"], label="MA60", color="#7c3aed", lw=1.1)
    ax.set_title(title, fontsize=14, weight="bold")
    ax.set_xlabel("Thời gian"); ax.set_ylabel("Giá")
    ax.grid(alpha=.2); ax.legend()
    fig.tight_layout(); fig.savefig(path, bbox_inches="tight"); plt.close(fig)

plot_market(stock_df, "FPT.VN: Giá đóng cửa và đường trung bình", "#0284c7", FIG_DIR / "fpt_market.png")
plot_market(gold_df, "GC=F: Giá vàng và đường trung bình", "#d97706", FIG_DIR / "gold_market.png")

# 4.4. Benchmark bằng chứng
fig, ax = plt.subplots(figsize=(10, 5.2))
benchmark.set_index("Phương pháp").plot.bar(ax=ax, color=["#38bdf8", "#fbbf24", "#a78bfa"])
ax.set_ylim(0, 1.08); ax.set_ylabel("Điểm [0, 1]")
ax.set_title("So sánh khả năng cung cấp bằng chứng", fontsize=14, weight="bold")
ax.grid(axis="y", alpha=.2); ax.tick_params(axis="x", rotation=0)
fig.tight_layout(); fig.savefig(FIG_DIR / "rag_benchmark.png", bbox_inches="tight"); plt.close(fig)





# 4.5. Biểu đồ kiểm thử theo thời gian cho FPT và vàng
def plot_market_predictions(predictions, metrics, title, path):
    fig, axes = plt.subplots(2, 1, figsize=(12, 7.2), gridspec_kw={"height_ratios": [2.2, 1]})
    axes[0].plot(predictions.index, predictions["Actual"], color="black", lw=1.5, label="Thực tế")
    axes[0].plot(predictions.index, predictions["Prediction"], color="#555555", lw=1.2,
                 linestyle="--", label=f"{metrics['Model']}")
    axes[0].plot(predictions.index, predictions["Persistence"], color="#999999", lw=1.0,
                 alpha=.8, label="Persistence")
    axes[0].set_title(f"{title} - dự báo giá đóng cửa phiên kế tiếp", weight="bold")
    axes[0].set_ylabel("Giá"); axes[0].legend(frameon=False); axes[0].grid(alpha=.18)
    residual = predictions["Actual"] - predictions["Prediction"]
    axes[1].axhline(0, color="black", lw=.9)
    axes[1].plot(residual.index, residual, color="#555555", lw=1.0)
    axes[1].set_ylabel("Sai số"); axes[1].set_xlabel("Thời gian"); axes[1].grid(alpha=.18)
    fig.text(.5, .01,
             f"MAE={metrics['MAE']:.3f} | RMSE={metrics['RMSE']:.3f} | "
             f"R²={metrics['R2']:.3f} | Direction={metrics['Direction_Accuracy']:.1f}%",
             ha="center", color="black")
    fig.tight_layout(rect=[0, .035, 1, 1]); fig.savefig(path, bbox_inches="tight", dpi=240); plt.close(fig)

plot_market_predictions(stock_predictions, stock_model_info, "FPT.VN", FIG_DIR / "stock_model.png")
plot_market_predictions(gold_predictions, gold_model_info, "GC=F", FIG_DIR / "gold_model.png")

# 4.6. Bảng mô tả dữ liệu dùng trong báo cáo
fig, ax = plt.subplots(figsize=(14, 3.3))
ax.axis("off")
table_view = data_profile_df[["Miền", "Nguồn", "Số dòng", "Số cột", "Phạm vi", "Biến đích/đơn vị"]].copy()
table = ax.table(cellText=table_view.values, colLabels=table_view.columns,
                 cellLoc="left", colLoc="left", loc="center",
                 colWidths=[.10, .27, .08, .07, .20, .22])
table.auto_set_font_size(False); table.set_fontsize(8.2); table.scale(1, 1.8)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#cbd5e1")
    if row == 0:
        cell.set_facecolor("#e5e7eb"); cell.set_text_props(color="black", weight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#f1f5f9")
ax.set_title("Mô tả ba nguồn dữ liệu tại thời điểm chạy", fontsize=14, weight="bold", pad=12)
fig.tight_layout(); fig.savefig(FIG_DIR / "data_profile.png", bbox_inches="tight"); plt.close(fig)

# 4.7. Đồ thị con theo từng miền
def draw_domain_graph(domain, title, filename, color):
    nodes = [n for n, a in kg.graph.nodes(data=True) if a.get("domain") == domain]
    sub = kg.graph.subgraph(nodes).copy()
    fig, ax = plt.subplots(figsize=(10.5, 6.3))
    pos = nx.spring_layout(sub, seed=RANDOM_SEED, k=1.35)
    nx.draw_networkx_edges(sub, pos, ax=ax, alpha=.5, arrows=True, arrowsize=16,
                           edge_color="#64748b", connectionstyle="arc3,rad=0.04")
    nx.draw_networkx_nodes(sub, pos, ax=ax, node_color=color, node_size=1600,
                           edgecolors="black", linewidths=1.5)
    nx.draw_networkx_labels(sub, pos, ax=ax, font_size=8)
    edge_labels = {(u, v): d.get("relation", k) for u, v, k, d in sub.edges(keys=True, data=True)}
    nx.draw_networkx_edge_labels(sub, pos, edge_labels=edge_labels, ax=ax, font_size=6,
                                 rotate=False, label_pos=.55)
    ax.set_title(title, fontsize=15, weight="bold"); ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG_DIR / filename, bbox_inches="tight"); plt.close(fig)

draw_domain_graph("diabetes", "Knowledge Graph miền bệnh tiểu đường", "kg_diabetes.png", "#fb7185")
draw_domain_graph("stock", "Knowledge Graph miền chứng khoán FPT", "kg_stock.png", "#38bdf8")
draw_domain_graph("gold", "Knowledge Graph miền giá vàng", "kg_gold.png", "#fbbf24")

# 4.8. Ảnh code: ảnh được tạo trực tiếp từ chính các đoạn code cốt lõi trong notebook.

def render_code_card(filename, title, source, accent="#000000"):
    source = textwrap.dedent(source).strip("\n")
    lines = source.splitlines()
    height = max(5.0, 0.31 * len(lines) + 1.25)
    fig, ax = plt.subplots(figsize=(12.8, height))
    fig.patch.set_facecolor("white"); ax.set_facecolor("white"); ax.axis("off")
    ax.add_patch(plt.Rectangle((0, .925), 1, .075, transform=ax.transAxes,
                               facecolor="#eeeeee", edgecolor="black", linewidth=1.0))
    ax.text(.025, .962, title, transform=ax.transAxes, va="center", ha="left",
            color="black", fontsize=12.5, weight="bold")
    ax.text(.975, .962, "Tieuluan02_06_Nhom2_Giangkt.ipynb", transform=ax.transAxes,
            va="center", ha="right", color="black", fontsize=8.5)
    numbered = "\n".join(f"{i:>2}  {line}" for i, line in enumerate(lines, 1))
    ax.text(.025, .895, numbered, transform=ax.transAxes, va="top", ha="left",
            family="Liberation Mono", fontsize=8.6, linespacing=1.35, color="black")
    fig.tight_layout(pad=.6)
    fig.savefig(FIG_DIR / filename, bbox_inches="tight", facecolor="white", dpi=240)
    plt.close(fig)

render_code_card("code_market_data.png", "Tải và tiền xử lý dữ liệu thị trường", r'''
def yahoo_chart(symbol, range_="2y", interval="1d"):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    response = requests.get(url, params={"range": range_, "interval": interval},
                            timeout=25, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    result = response.json()["chart"]["result"][0]
    quote = result["indicators"]["quote"][0]
    frame = pd.DataFrame(quote, index=pd.to_datetime(result["timestamp"], unit="s"))
    frame.columns = [c.title() for c in frame.columns]
    return frame[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])

def add_indicators(df):
    out = df.copy().sort_index()
    out["Return_1D"] = out["Close"].pct_change()
    out["MA20"] = out["Close"].rolling(20).mean()
    out["MA60"] = out["Close"].rolling(60).mean()
    out["Volatility20"] = out["Return_1D"].rolling(20).std() * np.sqrt(252)
    return out
''')

render_code_card("code_diabetes_training.png", "Tuning Random Forest bằng 5-fold CV", r'''
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rf_grid = {
    "n_estimators": [250, 400],
    "max_depth": [None, 8, 14],
    "min_samples_leaf": [2, 4, 6],
    "max_features": [0.7, 1.0],
}
search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=1),
    param_grid=rf_grid, scoring="neg_mean_absolute_error",
    cv=cv, n_jobs=1, refit=True,
)
search.fit(X_train, y_train)
diabetes_model = search.best_estimator_
diabetes_pred = diabetes_model.predict(X_test)
''', "#fb7185")



render_code_card("code_market_training.png", "TimeSeriesSplit cho FPT và giá vàng", r'''
supervised["Target_Next_Close"] = supervised["Close"].shift(-1)
split = int(len(supervised) * 0.8)
train, test = supervised.iloc[:split], supervised.iloc[split:]
tscv = TimeSeriesSplit(n_splits=5)
search = GridSearchCV(
    estimator, param_grid=grid,
    scoring="neg_mean_absolute_error",
    cv=tscv, n_jobs=1, refit=True,
)
search.fit(train[feature_cols], train["Target_Next_Close"])
prediction = search.best_estimator_.predict(test[feature_cols])
persistence = test["Close"].to_numpy()
''')

render_code_card("code_kg_core.png", "KnowledgeGraphCore: thực thể, quan hệ và multi-hop", r'''
class KnowledgeGraphCore:
    def __init__(self):
        self.graph = nx.MultiDiGraph()

    def add_entity(self, entity_id, entity_type="Entity", **properties):
        self.graph.add_node(entity_id, type=entity_type, **properties)

    def add_relation(self, subject, predicate, obj, weight=1.0, **properties):
        if subject not in self.graph: self.add_entity(subject)
        if obj not in self.graph: self.add_entity(obj)
        self.graph.add_edge(subject, obj, key=predicate,
                            relation=predicate, weight=float(weight), **properties)

    def multi_hop(self, start, max_hops=2):
        paths = []
        for target in self.graph.nodes:
            if target != start:
                paths.extend(nx.all_simple_paths(self.graph, start, target, cutoff=max_hops))
        return paths
''', "#fbbf24")

render_code_card("code_retriever.png", "VectorRetriever dùng TF-IDF và cosine similarity", r'''
class VectorRetriever:
    def __init__(self, docs):
        self.docs = docs
        self.vectorizer = TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), min_df=1,
            token_pattern=r"(?u)\b\w+\b"
        )
        self.matrix = self.vectorizer.fit_transform([d["text"] for d in docs])

    def search(self, query, top_k=3):
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix).ravel()
        order = np.argsort(scores)[::-1][:top_k]
        return [{**self.docs[i], "score": float(scores[i])} for i in order]
''', "#a78bfa")

render_code_card("code_graphrag.png", "GraphRAG: ghép bằng chứng văn bản và bộ ba", r'''
def graphrag_answer(query, top_k=3):
    docs = retriever.search(query, top_k=top_k)
    triples = graph_evidence(query)
    domain = detect_domain(query)
    domain_docs = [d for d in docs if d["domain"] in (domain, "method")]
    statements = [d["text"] for d in domain_docs[:2]]
    graph_text = [f"{s} -[{p}]-> {o}" for s, p, o in triples[:3]]
    answer = " ".join(statements)
    if graph_text:
        answer += " Quan hệ đồ thị: " + "; ".join(graph_text)
    return {"answer": answer, "documents": domain_docs, "triples": triples}
''', "#34d399")

render_code_card("code_deployment.png", "Xuất mô hình và chatbot RAG lên Vercel", r'''
model_config = export_linear_pipeline(model, feature_names)
model_json = json.dumps(model_config, ensure_ascii=False)
kb_json = json.dumps(health_kb, ensure_ascii=False)
html = (html_template
        .replace("__MODEL_CONFIG__", model_json)
        .replace("__HEALTH_KB__", kb_json))
(WEB_DIR / "index.html").write_text(html, encoding="utf-8")
# Triển khai: cd vercel_app && npx vercel --prod
''', "#60a5fa")


print("Đã tạo", len(list(FIG_DIR.glob("*.png"))), "hình trong", FIG_DIR)

Đã tạo 20 hình trong /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/artifacts_giangkt/figures


## 5. Sinh ứng dụng web và báo cáo PDF

Ứng dụng web tĩnh chạy ba mô hình suy luận và chatbot RAG ngay trên trình duyệt, nên triển khai nhanh trên Vercel mà không cần máy chủ Python.
Báo cáo PDF được tạo từ chính biến kết quả trong notebook để bảo đảm tái lập.

In [8]:
# 5.1. Ứng dụng web dự đoán và chatbot RAG chạy trực tiếp trên trình duyệt
def export_linear_pipeline(model, feature_names):
    # Đóng gói đúng pipeline StandardScaler + mô hình tuyến tính đã fit.
    if not isinstance(model, Pipeline) or "scale" not in model.named_steps:
        raise TypeError("Bản web hiện hỗ trợ pipeline tuyến tính có StandardScaler")
    scaler = model.named_steps["scale"]
    estimator = model.named_steps["model"]
    if not hasattr(estimator, "coef_"):
        raise TypeError("Mô hình triển khai phải có hệ số coef_")
    return {
        "features": list(feature_names),
        "mean": np.asarray(scaler.mean_, dtype=float).tolist(),
        "scale": np.asarray(scaler.scale_, dtype=float).tolist(),
        "coef": np.asarray(estimator.coef_, dtype=float).reshape(-1).tolist(),
        "intercept": float(np.asarray(estimator.intercept_).reshape(-1)[0]),
    }

diabetes_defaults = X.median().to_dict()
diabetes_ranges = {
    col: {"min": float(X[col].min()), "max": float(X[col].max()),
          "step": 0.001, "value": float(diabetes_defaults[col])}
    for col in X.columns
}

web_config = {
    "generated_at": RUN_AT.isoformat(),
    "student": {"name": STUDENT_NAME, "id": STUDENT_ID},
    "models": {
        "diabetes": {
            **export_linear_pipeline(diabetes_model, X.columns),
            "name": diabetes_metrics["Model"], "metrics": diabetes_metrics,
            "inputs": diabetes_ranges, "unit": "điểm tiến triển sau một năm",
        },
        "stock": {
            **export_linear_pipeline(stock_model, market_feature_cols),
            "name": stock_model_info["Model"], "metrics": stock_model_info,
            "latest": {k: float(v) for k, v in stock_latest_features.items()},
            "currency": stock_summary["currency"], "symbol": "FPT.VN",
            "as_of": stock_summary["end"],
        },
        "gold": {
            **export_linear_pipeline(gold_model, gold_feature_cols),
            "name": gold_model_info["Model"], "metrics": gold_model_info,
            "latest": {k: float(v) for k, v in gold_latest_features.items()},
            "currency": gold_summary["currency"] + "/oz", "symbol": "GC=F",
            "as_of": gold_summary["end"],
        },
    },
}

health_kb = [
    {
        "title": "Nguyên tắc chiếc đĩa",
        "keywords": ["bữa ăn", "ăn gì", "thực đơn", "chiếc đĩa", "rau", "đạm"],
        "answer": "Có thể bắt đầu bằng phương pháp chiếc đĩa: một nửa là rau không chứa nhiều tinh bột, một phần tư là đạm nạc và một phần tư là thực phẩm chứa carbohydrate. Ưu tiên nước lọc hoặc đồ uống ít năng lượng.",
        "source": "CDC - Diabetes Meal Planning; ADA - Diabetes Plate",
        "url": "https://www.cdc.gov/diabetes/healthy-eating/diabetes-meal-planning.html",
        "graph": "Rau không tinh bột -> chiếm 1/2 -> khẩu phần; carbohydrate -> chiếm 1/4 -> khẩu phần",
    },
    {
        "title": "Carbohydrate và đường huyết",
        "keywords": ["carb", "carbohydrate", "tinh bột", "cơm", "bún", "phở", "đường", "trái cây"],
        "answer": "Carbohydrate làm tăng đường huyết, nhưng không cần loại bỏ hoàn toàn. Nên giữ khẩu phần tương đối ổn định, ưu tiên ngũ cốc nguyên hạt, đậu, rau và trái cây nguyên quả; kết hợp cùng đạm và chất xơ để bữa ăn cân bằng hơn.",
        "source": "CDC - Diabetes Meal Planning",
        "url": "https://www.cdc.gov/diabetes/healthy-eating/diabetes-meal-planning.html",
        "graph": "Carbohydrate -> có thể làm tăng -> đường huyết; chất xơ -> hỗ trợ -> kiểm soát khẩu phần",
    },
    {
        "title": "Vận động thể lực",
        "keywords": ["vận động", "tập", "đi bộ", "thể dục", "bao nhiêu phút", "hoạt động"],
        "answer": "WHO khuyến nghị người trưởng thành hướng tới ít nhất 150 phút hoạt động thể lực mức vừa mỗi tuần. Có thể chia nhỏ thành các buổi phù hợp thể trạng; nếu đang dùng thuốc hạ đường huyết hoặc có biến chứng, nên hỏi nhân viên y tế trước khi đổi cường độ tập.",
        "source": "WHO - Diabetes fact sheet",
        "url": "https://www.who.int/news-room/fact-sheets/detail/diabetes",
        "graph": "Hoạt động thể lực -> hỗ trợ -> kiểm soát đường huyết; 150 phút/tuần -> là mục tiêu -> người trưởng thành",
    },
    {
        "title": "Theo dõi và tái khám",
        "keywords": ["đo đường", "đường huyết", "theo dõi", "tái khám", "khám", "hba1c", "thuốc"],
        "answer": "Lịch đo đường huyết và mục tiêu HbA1c phụ thuộc từng người, thuốc đang dùng và nguy cơ hạ đường huyết. Hãy ghi lại kết quả, bữa ăn và vận động để trao đổi trong lần tái khám; không tự đổi liều thuốc theo câu trả lời của chatbot.",
        "source": "WHO - Diabetes fact sheet; khuyến cáo an toàn của hệ thống",
        "url": "https://www.who.int/news-room/fact-sheets/detail/diabetes",
        "graph": "Theo dõi đường huyết -> cung cấp bằng chứng cho -> tái khám; thuốc -> cần chỉ định bởi -> nhân viên y tế",
    },
    {
        "title": "Hạ đường huyết và tình huống khẩn cấp",
        "keywords": ["hạ đường huyết", "run tay", "vã mồ hôi", "lú lẫn", "co giật", "bất tỉnh", "khẩn cấp"],
        "answer": "Run, vã mồ hôi, đói cồn cào hoặc lú lẫn có thể đi cùng hạ đường huyết. Nếu người bệnh bất tỉnh, co giật, khó thở hoặc không thể nuốt, cần gọi cấp cứu ngay và không cố cho ăn uống. Với triệu chứng nhẹ, làm theo kế hoạch xử trí đã được nhân viên y tế hướng dẫn.",
        "source": "CDC - Low Blood Sugar (Hypoglycemia); nguyên tắc an toàn cấp cứu",
        "url": "https://www.cdc.gov/diabetes/basics/low-blood-sugar-treatment.html",
        "graph": "Bất tỉnh/co giật -> yêu cầu -> cấp cứu; hạ đường huyết -> cần -> kế hoạch xử trí cá nhân",
    },
]

model_json = json.dumps(web_config, ensure_ascii=False).replace("</", "<\\/")
kb_json = json.dumps(health_kb, ensure_ascii=False).replace("</", "<\\/")

html_template = r'''<!doctype html>
<html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<meta name="description" content="Ứng dụng dự đoán ba miền và chatbot RAG sức khỏe bệnh tiểu đường">
<title>RAG HTTM - Dự đoán và Chatbot Sức khỏe</title>
<style>
:root{--navy:#10263f;--blue:#1769e0;--cyan:#dff6ff;--ink:#16202b;--muted:#617080;--line:#dce5ee;--paper:#fff;--bg:#f4f7fa;--good:#087f5b;--warn:#a15c00}
*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:var(--bg);color:var(--ink);font:16px/1.55 Inter,ui-sans-serif,system-ui,-apple-system,"Segoe UI",sans-serif}
a{color:inherit}.shell{max-width:1180px;margin:auto;padding:0 22px}.hero{background:radial-gradient(circle at 85% 10%,#2f85ff 0,transparent 30%),linear-gradient(135deg,#081a2d,#123f73);color:#fff;padding:70px 0 64px}.eyebrow{letter-spacing:.12em;text-transform:uppercase;font-size:.78rem;font-weight:800;color:#9fd7ff}.hero h1{font-size:clamp(2.2rem,5vw,4.4rem);line-height:1.05;max-width:900px;margin:.35em 0}.hero p{max-width:760px;color:#dcecff;font-size:1.08rem}.hero-actions{display:flex;gap:12px;flex-wrap:wrap;margin-top:26px}.btn{border:0;border-radius:12px;padding:12px 17px;font-weight:800;cursor:pointer;background:var(--blue);color:#fff;box-shadow:0 7px 18px #1769e033}.btn.secondary{background:#fff;color:var(--navy);text-decoration:none}.nav{position:sticky;top:0;z-index:10;background:#ffffffee;backdrop-filter:blur(12px);border-bottom:1px solid var(--line)}.nav .shell{display:flex;gap:20px;align-items:center;min-height:58px;overflow:auto}.nav a{text-decoration:none;font-weight:750;white-space:nowrap}.nav .brand{margin-right:auto;color:var(--blue)}
main{padding:24px 0 70px}.section{padding:38px 0}.section h2{font-size:clamp(1.55rem,3vw,2.25rem);margin:0 0 8px}.lead{margin:0 0 24px;color:var(--muted);max-width:850px}.grid{display:grid;grid-template-columns:repeat(3,1fr);gap:18px}.card{background:var(--paper);border:1px solid var(--line);border-radius:20px;padding:22px;box-shadow:0 10px 30px #10263f0b}.card h3{margin:0 0 6px}.badge{display:inline-flex;border-radius:999px;background:#e8f1ff;color:#1450a3;padding:5px 9px;font-size:.76rem;font-weight:800}.metric{font-size:1.7rem;font-weight:850;color:var(--blue);margin:12px 0 0}.small{font-size:.86rem;color:var(--muted)}.tabs{display:flex;gap:8px;flex-wrap:wrap;margin:20px 0}.tab{border:1px solid var(--line);background:#fff;padding:10px 14px;border-radius:999px;font-weight:800;cursor:pointer}.tab.active{background:var(--navy);color:#fff}.panel{display:none}.panel.active{display:block}.form-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:13px}.field label{display:block;font-size:.86rem;font-weight:750;margin-bottom:5px}.field input{width:100%;border:1px solid #bac8d5;border-radius:10px;padding:10px 11px;font:inherit;background:#fff}.result{margin-top:18px;border-left:5px solid var(--blue);background:#edf5ff;border-radius:10px;padding:14px 16px}.result b{font-size:1.45rem}.note{border-radius:14px;padding:13px 15px;background:#fff6e7;color:#6f4200;border:1px solid #f2d19b;margin-top:14px}
.chat-layout{display:grid;grid-template-columns:1.5fr .75fr;gap:18px}.chat{height:470px;display:flex;flex-direction:column;padding:0;overflow:hidden}.messages{padding:20px;overflow:auto;flex:1;background:linear-gradient(#f8fbff,#fff)}.bubble{max-width:86%;padding:12px 14px;border-radius:16px;margin:0 0 12px;white-space:pre-line}.bot{background:#eaf3ff;border-bottom-left-radius:4px}.user{background:var(--navy);color:#fff;margin-left:auto;border-bottom-right-radius:4px}.sources{font-size:.78rem;color:#526577;margin-top:7px}.composer{display:flex;gap:8px;border-top:1px solid var(--line);padding:12px;background:#fff}.composer input{flex:1;border:1px solid #bac8d5;border-radius:11px;padding:11px;font:inherit}.chips{display:flex;gap:7px;flex-wrap:wrap;margin-top:13px}.chip{border:1px solid #bcd2eb;background:#fff;color:#174b82;border-radius:999px;padding:7px 10px;cursor:pointer}.evidence li{margin-bottom:9px}.status{display:flex;gap:8px;align-items:center;color:var(--good);font-weight:750}.dot{width:9px;height:9px;border-radius:50%;background:var(--good);box-shadow:0 0 0 4px #0ca67822}.footer{border-top:1px solid var(--line);padding:28px 0 40px;color:var(--muted)}
@media(max-width:860px){.grid{grid-template-columns:1fr}.chat-layout{grid-template-columns:1fr}.form-grid{grid-template-columns:1fr}.nav .brand{display:none}.hero{padding:54px 0}.chat{height:520px}}
</style></head><body>
<header class="hero"><div class="shell"><div class="eyebrow">Knowledge Graph · Machine Learning · RAG</div><h1>Dự đoán dữ liệu và hỏi đáp sức khỏe trong một ứng dụng</h1><p>Ba mô hình được huấn luyện trong notebook cho bệnh tiểu đường, cổ phiếu FPT và giá vàng. Chatbot sức khỏe truy xuất bằng chứng từ kho tri thức có nguồn, không trả lời kiểu đoán mò.</p><div class="hero-actions"><a class="btn secondary" href="#predict">Thử dự đoán</a><a class="btn" href="#chatbot">Mở chatbot</a></div><p class="small">Kiều Trường Giang · B23DCCN254</p></div></header>
<nav class="nav"><div class="shell"><a class="brand" href="#">RAG HTTM</a><a href="#overview">Tổng quan</a><a href="#predict">Dự đoán</a><a href="#chatbot">Chatbot tiểu đường</a><a href="__GITHUB__" target="_blank" rel="noreferrer">GitHub</a></div></nav>
<main class="shell">
<section id="overview" class="section"><h2>Mô hình đã được huấn luyện</h2><p class="lead">Kết quả dưới đây lấy trực tiếp từ lần Run All notebook. Tiểu đường dùng K-fold; FPT và vàng dùng TimeSeriesSplit và tập kiểm thử nằm sau tập huấn luyện theo thời gian.</p><div class="grid" id="metricCards"></div></section>
<section id="predict" class="section"><h2>Khu vực dự đoán trực tiếp</h2><p class="lead">Thay đổi đầu vào và bấm dự đoán. Trình duyệt áp dụng đúng StandardScaler, hệ số và intercept của pipeline đã fit; không dùng số ngẫu nhiên hay kết quả viết sẵn.</p><div class="tabs"><button class="tab active" data-target="diabetes">Tiểu đường</button><button class="tab" data-target="stock">FPT phiên kế tiếp</button><button class="tab" data-target="gold">Vàng phiên kế tiếp</button></div>
<div class="card panel active" id="panel-diabetes"><h3>Ước lượng chỉ số tiến triển bệnh sau một năm</h3><p class="small">Bộ dữ liệu scikit-learn gồm các biến đã chuẩn hóa. Đây là điểm nghiên cứu hồi quy, không phải xác suất mắc bệnh hay chẩn đoán.</p><div class="form-grid" id="diabetesForm"></div><button class="btn" onclick="runDiabetes()" style="margin-top:16px">Dự đoán</button><div class="result" id="diabetesResult">Nhập thông tin rồi bấm dự đoán.</div></div>
<div class="card panel" id="panel-stock"><h3>Dự báo giá đóng cửa FPT.VN phiên kế tiếp</h3><p class="small" id="stockDate"></p><div class="form-grid" id="stockForm"></div><button class="btn" onclick="runMarket('stock')" style="margin-top:16px">Dự đoán FPT</button><div class="result" id="stockResult">Các trường được điền bằng quan sát mới nhất khi notebook chạy.</div><div class="note">Dự báo một bước phục vụ thực nghiệm học máy, không phải khuyến nghị mua bán.</div></div>
<div class="card panel" id="panel-gold"><h3>Dự báo giá đóng cửa vàng GC=F phiên kế tiếp</h3><p class="small" id="goldDate"></p><div class="form-grid" id="goldForm"></div><button class="btn" onclick="runMarket('gold')" style="margin-top:16px">Dự đoán vàng</button><div class="result" id="goldResult">Các trường được điền bằng quan sát mới nhất khi notebook chạy.</div><div class="note">Giá vàng biến động theo nhiều yếu tố ngoài mô hình; kết quả không phải khuyến nghị đầu tư.</div></div></section>
<section id="chatbot" class="section"><h2>Chatbot Sức khỏe & Dinh dưỡng Bệnh Tiểu đường</h2><p class="lead">Trợ lý dùng truy xuất từ kho tri thức nhỏ và quan hệ KG. Mỗi câu trả lời kèm nguồn và đường bằng chứng thay vì tự chẩn đoán.</p><div class="chat-layout"><div class="card chat"><div class="messages" id="messages"><div class="bubble bot">Chào bạn, mình có thể giải thích cách sắp xếp bữa ăn, carbohydrate, vận động và theo dõi đường huyết cho người bệnh tiểu đường. Bạn muốn hỏi điều gì?</div></div><div class="composer"><input id="chatInput" placeholder="Ví dụ: Một bữa ăn nên chia thế nào?" aria-label="Câu hỏi"><button class="btn" id="sendBtn">Gửi</button></div></div><aside class="card evidence"><div class="status"><span class="dot"></span>Kho tri thức đang hoạt động</div><h3 style="margin-top:18px">Câu hỏi gợi ý</h3><div class="chips"><button class="chip">Gợi ý một bữa ăn</button><button class="chip">Nên ăn tinh bột thế nào?</button><button class="chip">Vận động bao nhiêu?</button><button class="chip">Khi nào cần cấp cứu?</button></div><h3 style="margin-top:22px">Nguyên tắc trả lời</h3><ul><li>Truy xuất đoạn phù hợp bằng từ khóa có trọng số.</li><li>Hiển thị nguồn CDC, WHO hoặc ADA.</li><li>Không chẩn đoán, kê đơn hay đổi liều thuốc.</li></ul></aside></div></section>
</main><footer class="footer"><div class="shell">Đồ án Knowledge Graph và RAG · Kiều Trường Giang · Dữ liệu và mô hình được tạo từ Tieuluan02_06_Nhom2_Giangkt.ipynb.</div></footer>
<script>
const CONFIG=__MODEL_CONFIG__;
const HEALTH_KB=__HEALTH_KB__;
const labels={age:'Tuổi (chuẩn hóa)',sex:'Giới tính (chuẩn hóa)',bmi:'BMI (chuẩn hóa)',bp:'Huyết áp (chuẩn hóa)',s1:'S1 - cholesterol toàn phần',s2:'S2 - LDL',s3:'S3 - HDL',s4:'S4 - tỉ lệ cholesterol/HDL',s5:'S5 - log triglyceride',s6:'S6 - glucose',Close:'Giá đóng cửa',Close_Lag1:'Giá đóng cửa phiên trước',Return_1D:'Lợi suất 1 ngày',Return_Lag1:'Lợi suất trễ 1 ngày',MA20:'Trung bình 20 phiên',MA60:'Trung bình 60 phiên',Volatility20:'Biến động 20 phiên',RSI14:'RSI 14',Volume_Change:'Thay đổi khối lượng',Range_Pct:'Biên độ cao-thấp',Open_Close_Pct:'Thay đổi mở-đóng'};
function predict(model, values){return model.intercept+model.coef.reduce((sum,c,i)=>sum+c*((values[model.features[i]]-model.mean[i])/model.scale[i]),0)}
function num(x,d=4){return Number(x).toLocaleString('vi-VN',{maximumFractionDigits:d})}
function render(){const models=CONFIG.models;document.getElementById('metricCards').innerHTML=[['Tiểu đường',models.diabetes],['FPT.VN',models.stock],['Vàng GC=F',models.gold]].map(([title,m])=>`<article class="card"><span class="badge">${m.name}</span><h3>${title}</h3><div class="metric">R² ${num(m.metrics.R2,3)}</div><p class="small">MAE test: ${num(m.metrics.MAE,2)} · CV MAE: ${num(m.metrics.CV_MAE,2)}</p></article>`).join('');
const df=document.getElementById('diabetesForm');Object.entries(models.diabetes.inputs).forEach(([k,v])=>df.insertAdjacentHTML('beforeend',`<div class="field"><label for="d-${k}">${labels[k]||k}</label><input id="d-${k}" type="number" min="${v.min}" max="${v.max}" step="${v.step}" value="${v.value.toFixed(4)}"></div>`));
['stock','gold'].forEach(kind=>{const m=models[kind], form=document.getElementById(kind+'Form');document.getElementById(kind+'Date').textContent=`Mã ${m.symbol} · dữ liệu đến ${m.as_of} · đơn vị ${m.currency}`;['Close','MA20','MA60','RSI14','Return_1D','Volatility20'].forEach(k=>form.insertAdjacentHTML('beforeend',`<div class="field"><label for="${kind}-${k}">${labels[k]}</label><input id="${kind}-${k}" type="number" step="any" value="${m.latest[k].toFixed(6)}"></div>`))})}
function runDiabetes(){const m=CONFIG.models.diabetes,v={};m.features.forEach(k=>v[k]=Number(document.getElementById('d-'+k).value));const y=predict(m,v);document.getElementById('diabetesResult').innerHTML=`Kết quả ước lượng: <b>${num(y,1)} điểm</b><br><span class="small">Khoảng quan sát của bộ dữ liệu là điểm tiến triển bệnh sau một năm. Kết quả không thay cho xét nghiệm hay chẩn đoán.</span>`}
function runMarket(kind){const m=CONFIG.models[kind],v={...m.latest};['Close','MA20','MA60','RSI14','Return_1D','Volatility20'].forEach(k=>v[k]=Number(document.getElementById(`${kind}-${k}`).value));const y=predict(m,v),diff=(y-v.Close)/v.Close*100;document.getElementById(kind+'Result').innerHTML=`Giá đóng cửa dự báo: <b>${num(y,2)} ${m.currency}</b><br><span class="small">Chênh lệch so với giá nhập: ${diff>=0?'+':''}${num(diff,2)}%</span>`}
document.querySelectorAll('.tab').forEach(b=>b.addEventListener('click',()=>{document.querySelectorAll('.tab').forEach(x=>x.classList.remove('active'));document.querySelectorAll('.panel').forEach(x=>x.classList.remove('active'));b.classList.add('active');document.getElementById('panel-'+b.dataset.target).classList.add('active')}));
function normalize(s){return s.toLowerCase().normalize('NFD').replace(/[\u0300-\u036f]/g,'')}
function retrieve(q){const n=normalize(q);return HEALTH_KB.map(doc=>({...doc,score:doc.keywords.reduce((s,k)=>s+(n.includes(normalize(k))?3+normalize(k).split(' ').length:0),0)})).sort((a,b)=>b.score-a.score)}
function ask(q){const urgent=['bat tinh','co giat','kho tho','dau nguc','khong the nuot'].some(x=>normalize(q).includes(x));const ranked=retrieve(q),doc=ranked[0].score?ranked[0]:HEALTH_KB[0];let answer=urgent?'Các dấu hiệu bạn mô tả có thể là tình huống khẩn cấp. Hãy gọi cấp cứu hoặc đưa người bệnh đến cơ sở y tế ngay; không cố cho ăn uống nếu người bệnh bất tỉnh hoặc không thể nuốt.':doc.answer;return {answer,doc}}
function addMessage(text,who,meta=''){const el=document.createElement('div');el.className='bubble '+who;el.innerHTML=text+(meta?`<div class="sources">${meta}</div>`:'');document.getElementById('messages').appendChild(el);el.scrollIntoView({behavior:'smooth',block:'end'})}
function send(){const input=document.getElementById('chatInput'),q=input.value.trim();if(!q)return;addMessage(q,'user');input.value='';setTimeout(()=>{const r=ask(q);addMessage(r.answer,'bot',`Nguồn: <a href="${r.doc.url}" target="_blank" rel="noreferrer">${r.doc.source}</a><br>Bằng chứng KG: ${r.doc.graph}`)},180)}
document.getElementById('sendBtn').addEventListener('click',send);document.getElementById('chatInput').addEventListener('keydown',e=>{if(e.key==='Enter')send()});document.querySelectorAll('.chip').forEach(c=>c.addEventListener('click',()=>{document.getElementById('chatInput').value=c.textContent;send()}));render();
</script></body></html>'''

html = (html_template
        .replace("__MODEL_CONFIG__", model_json)
        .replace("__HEALTH_KB__", kb_json)
        .replace("__GITHUB__", GITHUB_URL))
(WEB_DIR / "index.html").write_text(html, encoding="utf-8")
(WEB_DIR / "vercel.json").write_text(
    json.dumps({"cleanUrls": True, "trailingSlash": False}, indent=2), encoding="utf-8"
)
(WEB_DIR / "README.txt").write_text(
    "Ứng dụng dự đoán ba miền và chatbot RAG được tạo bởi Tieuluan02_06_Nhom2_Giangkt.ipynb.\n"
    f"Production: {DEPLOY_URL}\n", encoding="utf-8"
)

# Hình minh họa giao diện để đưa vào báo cáo với độ phân giải cao.
fig, ax = plt.subplots(figsize=(13.2, 7.4), dpi=200)
ax.set_xlim(0, 13.2); ax.set_ylim(0, 7.4); ax.axis("off")
ax.add_patch(plt.Rectangle((0, 0), 13.2, 7.4, color="#f4f7fa"))
ax.add_patch(plt.Rectangle((0, 5.75), 13.2, 1.65, color="#10263f"))
ax.text(.55, 6.78, "RAG HTTM", color="#9fd7ff", fontsize=10, weight="bold")
ax.text(.55, 6.22, "Dự đoán dữ liệu và Chatbot Sức khỏe", color="white", fontsize=21, weight="bold")
ax.text(.55, 5.92, "Ba mô hình đã huấn luyện · RAG có nguồn · chạy trực tiếp trên web", color="#dcecff", fontsize=10)
for i, (title, val) in enumerate([
    ("TIỂU ĐƯỜNG", f"R² {diabetes_metrics['R2']:.3f}"),
    ("FPT.VN", f"R² {stock_model_info['R2']:.3f}"),
    ("VÀNG GC=F", f"R² {gold_model_info['R2']:.3f}"),
]):
    x=.55+i*4.2
    ax.add_patch(plt.Rectangle((x, 4.35), 3.75, 1.05, facecolor="white", edgecolor="#dce5ee", linewidth=1))
    ax.text(x+.2, 5.08, title, fontsize=8, color="#617080", weight="bold")
    ax.text(x+.2, 4.62, val, fontsize=16, color="#1769e0", weight="bold")
ax.add_patch(plt.Rectangle((.55, .45), 7.45, 3.55, facecolor="white", edgecolor="#dce5ee"))
ax.text(.82, 3.65, "DỰ ĐOÁN TRỰC TIẾP", fontsize=11, color="#10263f", weight="bold")
for j, label in enumerate(["Giá đóng cửa", "MA20", "MA60", "RSI14"]):
    xx=.82+(j%2)*3.45; yy=2.95-(j//2)*1.0
    ax.text(xx, yy+0.35, label, fontsize=8, color="#617080")
    ax.add_patch(plt.Rectangle((xx, yy-.05), 2.95, .36, facecolor="#f8fbff", edgecolor="#bac8d5"))
ax.add_patch(plt.Rectangle((.82, .72), 1.55, .48, facecolor="#1769e0", edgecolor="none"))
ax.text(1.595, .96, "Dự đoán", ha="center", va="center", color="white", fontsize=9, weight="bold")
ax.add_patch(plt.Rectangle((8.35, .45), 4.3, 3.55, facecolor="white", edgecolor="#dce5ee"))
ax.text(8.62, 3.65, "CHATBOT TIỂU ĐƯỜNG", fontsize=11, color="#10263f", weight="bold")
ax.add_patch(plt.Rectangle((8.62, 2.5), 3.35, .72, facecolor="#eaf3ff", edgecolor="none"))
ax.text(8.82, 2.86, "Mình có thể hỗ trợ bữa ăn,\ncarbohydrate và vận động.", fontsize=8.5, va="center")
ax.add_patch(plt.Rectangle((9.35, 1.7), 2.62, .48, facecolor="#10263f", edgecolor="none"))
ax.text(9.55, 1.94, "Nên ăn tinh bột thế nào?", fontsize=7.7, color="white", va="center")
ax.add_patch(plt.Rectangle((8.62, .72), 3.35, .52, facecolor="#f8fbff", edgecolor="#bac8d5"))
ax.text(8.82, .98, "Nhập câu hỏi...", fontsize=8, color="#617080", va="center")
fig.tight_layout(pad=0); fig.savefig(FIG_DIR / "web_app_preview.png", dpi=220, bbox_inches="tight"); plt.close(fig)
print("Ứng dụng web:", WEB_DIR / "index.html")
print("Cấu hình đã xuất:", {k: v["name"] for k, v in web_config["models"].items()})


Ứng dụng web: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/vercel_app/index.html
Cấu hình đã xuất: {'diabetes': 'Ridge', 'stock': 'Ridge', 'gold': 'Ridge'}


In [9]:
# 5.2. Tạo báo cáo PDF chi tiết
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.platypus import (
    BaseDocTemplate, Frame, PageTemplate, Paragraph, Spacer, PageBreak,
    Image, Table, TableStyle, KeepTogether
)
from reportlab.platypus.tableofcontents import TableOfContents


FONT_DIR = Path("/usr/share/fonts/truetype/liberation")
pdfmetrics.registerFont(TTFont("TNR", str(FONT_DIR / "LiberationSerif-Regular.ttf")))
pdfmetrics.registerFont(TTFont("TNR-Bold", str(FONT_DIR / "LiberationSerif-Bold.ttf")))
pdfmetrics.registerFont(TTFont("TNR-Italic", str(FONT_DIR / "LiberationSerif-Italic.ttf")))

PAGE_W, PAGE_H = A4
ink, pale, line = colors.black, colors.HexColor("#f2f2f2"), colors.HexColor("#777777")
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="VNBody", parent=styles["BodyText"], fontName="TNR", fontSize=12,
                          leading=17, alignment=TA_JUSTIFY, spaceAfter=8, textColor=ink))
styles.add(ParagraphStyle(name="VNTitle", parent=styles["Title"], fontName="TNR-Bold", fontSize=22,
                          leading=28, alignment=TA_CENTER, textColor=ink, spaceAfter=12))
styles.add(ParagraphStyle(name="VNChapter", parent=styles["Heading1"], fontName="TNR-Bold", fontSize=16,
                          leading=21, textColor=ink, spaceAfter=12, keepWithNext=True))
styles.add(ParagraphStyle(name="VNHeading", parent=styles["Heading2"], fontName="TNR-Bold", fontSize=13,
                          leading=18, textColor=ink, spaceAfter=9, keepWithNext=True))
styles.add(ParagraphStyle(name="VNSmall", parent=styles["BodyText"], fontName="TNR", fontSize=9,
                          leading=12, textColor=ink))
styles.add(ParagraphStyle(name="VNCaption", parent=styles["BodyText"], fontName="TNR-Italic", fontSize=9,
                          leading=12, alignment=TA_CENTER, textColor=ink, spaceAfter=8))
styles.add(ParagraphStyle(name="VNQuote", parent=styles["BodyText"], fontName="TNR", fontSize=11,
                          leading=16, leftIndent=14, rightIndent=14, borderColor=ink, borderWidth=.6,
                          borderPadding=8, backColor=pale, spaceAfter=10, textColor=ink))


class ReportDoc(BaseDocTemplate):
    def __init__(self, filename):
        super().__init__(filename, pagesize=A4, rightMargin=1.8*cm, leftMargin=2.2*cm,
                         topMargin=1.7*cm, bottomMargin=1.8*cm,
                         title="Knowledge Graph và RAG", author=STUDENT_NAME)
        frame = Frame(self.leftMargin, self.bottomMargin, self.width, self.height, id="normal")
        self.addPageTemplates(PageTemplate(id="body", frames=frame, onPage=self._header_footer))

    def _header_footer(self, canvas, doc):
        canvas.saveState()
        canvas.setFillColor(colors.black); canvas.setStrokeColor(colors.black)
        if doc.page == 1:
            canvas.setLineWidth(1.0)
            canvas.rect(0.9*cm, 0.9*cm, PAGE_W-1.8*cm, PAGE_H-1.8*cm)
            canvas.setLineWidth(.35)
            canvas.rect(1.1*cm, 1.1*cm, PAGE_W-2.2*cm, PAGE_H-2.2*cm)
            canvas.setFont("TNR-Bold", 28)
            canvas.drawCentredString(PAGE_W/2, PAGE_H-7.0*cm, "PTIT")
        else:
            canvas.setLineWidth(.35)
            canvas.line(2.2*cm, PAGE_H-1.2*cm, PAGE_W-1.8*cm, PAGE_H-1.2*cm)
            canvas.setFont("TNR", 8)
            canvas.drawString(2.2*cm, PAGE_H-0.95*cm, "CHƯƠNG 4 - KNOWLEDGE GRAPH VÀ RAG")
            canvas.drawRightString(PAGE_W-1.8*cm, PAGE_H-0.95*cm, f"{STUDENT_NAME} - {STUDENT_ID}")
            canvas.drawCentredString(PAGE_W/2, 0.9*cm, str(doc.page - 1))
        canvas.restoreState()

    def afterFlowable(self, flowable):
        if not isinstance(flowable, Paragraph):
            return
        title = flowable.getPlainText()
        style_name = flowable.style.name
        if style_name not in {"VNChapter", "VNHeading"}:
            return
        if title.startswith("4.") or title.startswith("PHỤ LỤC") or title == "TÀI LIỆU THAM KHẢO":
            level = 0 if style_name == "VNChapter" else 1
            self.notify("TOCEntry", (level, title, self.page - 1))

def P(text, style="VNBody"):
    clean = str(text).replace("—", "-").replace("–", "-").replace("‑", "-")
    return Paragraph(clean, styles[style])

def report_table(data, widths=None, header=True, font_size=9.0):
    table = Table(data, colWidths=widths, repeatRows=1 if header else 0, hAlign="LEFT")
    table.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,-1), "TNR"), ("FONTSIZE", (0,0), (-1,-1), font_size),
        ("LEADING", (0,0), (-1,-1), font_size+3), ("TEXTCOLOR", (0,0), (-1,-1), colors.black),
        ("VALIGN", (0,0), (-1,-1), "TOP"), ("GRID", (0,0), (-1,-1), .45, colors.black),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#f7f7f7")]),
        ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#e5e5e5") if header else colors.white),
        ("FONTNAME", (0,0), (-1,0), "TNR-Bold"), ("LEFTPADDING", (0,0), (-1,-1), 6),
        ("RIGHTPADDING", (0,0), (-1,-1), 6), ("TOPPADDING", (0,0), (-1,-1), 6),
        ("BOTTOMPADDING", (0,0), (-1,-1), 6),
    ]))
    return table

def report_image(path, max_width=15.5*cm, max_height=11.2*cm):
    image = Image(str(path))
    image._restrictSize(max_width, max_height)
    return image

story = []

# Bìa
story += [Spacer(1, 1.1*cm), P(SCHOOL, "VNHeading"), P(FACULTY, "VNHeading"), Spacer(1, 2.2*cm),
          P("CHƯƠNG 04", "VNTitle"), P("KNOWLEDGE GRAPH VÀ RAG", "VNTitle"), Spacer(1, 1.6*cm),
          P(f"<b>Học phần:</b> {COURSE}"), P("<b>Giảng viên hướng dẫn:</b> Thầy Trần Đình Quế"),
          P(f"<b>Sinh viên thực hiện:</b> {STUDENT_NAME}"), P(f"<b>Mã sinh viên:</b> {STUDENT_ID}"),
              P(f"<b>GitHub:</b> <link href='{GITHUB_URL}' color='black'>{GITHUB_URL}</link>"),
              P(f"<b>Web demo:</b> <link href='{DEPLOY_URL}' color='black'>{DEPLOY_URL}</link>"),
          Spacer(1, 4.0*cm), P("Hà Nội — 2026", "VNHeading"), PageBreak()]



story.append(P("MỤC LỤC", "VNChapter"))
toc = TableOfContents()
toc.levelStyles = [
    ParagraphStyle(name="TOC0", fontName="TNR-Bold", fontSize=11.5, leading=17,
                   leftIndent=0, firstLineIndent=0, textColor=colors.black, spaceBefore=4),
    ParagraphStyle(name="TOC1", fontName="TNR", fontSize=10.5, leading=15,
                   leftIndent=18, firstLineIndent=0, textColor=colors.black),
]
story += [toc, Spacer(1, .45*cm),
          P(f"<b>Repository:</b> <link href='{GITHUB_URL}' color='black'>{GITHUB_URL}</link>"),
          P(f"<b>Web demo:</b> <link href='{DEPLOY_URL}' color='black'>{DEPLOY_URL}</link>"), PageBreak()]

pages = [
    ("4.1. MỞ ĐẦU", [
        "Knowledge Graph (KG) và Retrieval-Augmented Generation (RAG) xuất phát từ hai truyền thống khác nhau. KG nhấn mạnh biểu diễn tường minh: thực thể nào tồn tại, chúng liên hệ ra sao và ràng buộc nào phải được giữ. RAG xuất phát từ nhu cầu cho mô hình sinh tiếp cận một kho tri thức ngoài tại thời điểm trả lời. Khi ghép hai hướng này, hệ thống vừa có khả năng tìm kiếm mềm theo ngôn ngữ, vừa có cấu trúc để kiểm tra đường suy luận.",
        "Bài làm lựa chọn kiến trúc có thể chạy trọn trong một notebook. TF-IDF đảm nhiệm truy xuất văn bản, NetworkX lưu đồ thị, còn bộ tổng hợp ghép bằng chứng tìm được thành câu trả lời. Dữ liệu FPT.VN và GC=F được tải trực tiếp khi chạy; dữ liệu tiểu đường lấy từ scikit-learn.",
        "Ba miền được chọn tạo ra phép thử đủ khác biệt. Y tế yêu cầu diễn đạt thận trọng; chứng khoán là chuỗi thời gian nội địa; vàng chịu tác động của yếu tố vĩ mô và được niêm yết quốc tế. Một pipeline dùng chung phải giữ được ngữ cảnh riêng của từng miền mà không trộn lẫn bằng chứng."], "architecture.png"),
    ("4.1.1. LỊCH SỬ PHÁT TRIỂN CỦA KNOWLEDGE GRAPH", [
        "Ý tưởng nối các khái niệm thành mạng ngữ nghĩa đã xuất hiện từ những nghiên cứu ban đầu về trí nhớ và biểu diễn tri thức. Sau đó, hệ chuyên gia đưa luật IF–THEN vào các miền hẹp. Bước ngoặt của Web ngữ nghĩa là việc chuẩn hóa RDF, RDFS, OWL và SPARQL, khiến bộ ba chủ thể–vị từ–đối tượng trở thành một ngôn ngữ trao đổi máy đọc được [1], [2].",
        "Năm 2012, cách gọi Knowledge Graph được phổ biến rộng rãi qua công cụ tìm kiếm của Google. Trong doanh nghiệp, KG tiếp tục được dùng cho danh mục sản phẩm, phát hiện gian lận, quản trị dữ liệu chủ và gợi ý nội dung. Điểm chung không phải một loại cơ sở dữ liệu cụ thể, mà là cách coi quan hệ giữa các thực thể là công dân hạng nhất.",
        "Ở bài này, đồ thị được cài bằng Labeled Property Graph đơn giản. Mỗi nút có type và domain; mỗi cạnh có nhãn quan hệ. Lựa chọn này gần với cách Neo4j tổ chức dữ liệu và thuận tiện cho trực quan hóa, dù chưa khai thác đầy đủ suy luận ontology như OWL."], None),
    ("4.1.1. LỊCH SỬ PHÁT TRIỂN CỦA RAG", [
        "Trước RAG, các hệ hỏi đáp mở thường tách bộ tìm kiếm tài liệu và bộ đọc hiểu. REALM đưa cơ chế truy xuất vào quá trình tiền huấn luyện [3]. Lewis và cộng sự sau đó định danh kiến trúc Retrieval-Augmented Generation, kết hợp bộ truy xuất dense với mô hình sinh seq2seq cho nhiệm vụ giàu tri thức [4].",
        "Sự phổ biến của mô hình ngôn ngữ lớn khiến RAG trở thành một mẫu triển khai thực dụng: tài liệu có thể cập nhật mà không cần huấn luyện lại toàn bộ mô hình, đồng thời câu trả lời giữ được liên kết về nguồn. Tuy nhiên, vector search thường ưu tiên đoạn giống câu hỏi và có thể bỏ qua quan hệ gián tiếp. GraphRAG giải quyết phần thiếu này bằng việc xây thực thể, quan hệ, cụm và tóm tắt theo cấu trúc [5].",
        "Nguyên mẫu trong notebook không cố tái tạo toàn bộ Microsoft GraphRAG. Nó giữ hai ý cốt lõi: truy xuất đồng thời từ văn bản và đồ thị, sau đó đưa cả hai nhóm bằng chứng vào bước tổng hợp."], None),
    ("4.1.2. VAI TRÒ VÀ Ý NGHĨA TRONG AI", [
        "KG giúp mô hình có từ điển thực thể thống nhất, quan hệ có tên và đường đi có thể kiểm tra. RAG giúp tri thức ngoài đi vào ngữ cảnh trả lời đúng thời điểm. Kết hợp hai thành phần làm giảm nguy cơ câu trả lời chỉ dựa trên tương đồng bề mặt, nhưng không đồng nghĩa loại bỏ hoàn toàn sai sót.",
        "Trong y tế, giá trị quan trọng nhất là provenance: người đọc phải biết nhận xét dựa trên biến nào và nguồn nào. Trong tài chính, dữ liệu có tính thời điểm nên ngày quan sát phải đi cùng con số. Trong cả hai trường hợp, câu trả lời nên phân biệt sự kiện đo được, quan hệ thường gặp và kết luận mang tính dự báo.",
        "Ý nghĩa học thuật của bài nằm ở khả năng chạy lại. Người chấm có thể Run All, xem dữ liệu thô đã lưu, đối chiếu bộ ba, kiểm tra tài liệu được truy xuất và mở PDF được sinh từ cùng một trạng thái chạy."], None),
    ("4.1.3. PHẠM VI, CÂU HỎI VÀ ĐÓNG GÓP", [
        "Bài đặt ba câu hỏi: (1) một cấu trúc KG thống nhất có mô tả được ba miền khác nhau không; (2) truy xuất lai có cung cấp nhiều bằng chứng hơn vector search đơn lẻ không; và (3) có thể đóng gói toàn bộ quy trình trong một notebook duy nhất không.",
        "Đóng góp thực hành gồm lớp KnowledgeGraphCore, bộ tải Yahoo Finance không cần API key, các bước tạo chỉ báo, VectorRetriever, truy xuất đồ thị theo miền, đánh giá precision@3, ứng dụng dự đoán, chatbot sức khỏe có nguồn và bộ sinh PDF. Mọi tệp phụ đều được tạo lại từ notebook.",
        "Ba nguồn dữ liệu được mô tả bằng số dòng, số cột, khoảng thời gian, đơn vị và biến đích. Bảng được sinh trực tiếp từ DataFrame sau xử lý nên khớp với lần chạy hiện tại."], "data_profile.png"),
    ("4.2. KNOWLEDGE GRAPH – KNOWLEDGE BASE", [
        "Knowledge Base trong bài gồm hai lớp. Lớp văn bản giữ các đoạn mô tả dữ liệu, phương pháp và kết quả theo từng miền. Lớp đồ thị giữ thực thể cùng các quan hệ có hướng. Hai lớp dùng chung nhãn domain để ngăn truy xuất nhầm giữa tiểu đường, cổ phiếu và vàng.",
        f"Đồ thị tại lần chạy này có {kg.graph.number_of_nodes()} nút và {kg.graph.number_of_edges()} cạnh. Kích thước nhỏ giúp người đọc kiểm tra thủ công. Khi mở rộng, cùng schema có thể chuyển sang Neo4j hoặc một triple store mà không thay đổi ý nghĩa bộ ba.",
        "Knowledge Base không chỉ chứa kiến thức tĩnh. Các nút quan sát như giá FPT hoặc giá vàng gần nhất được tạo từ dữ liệu tải về. Cách làm này nối câu trả lời với đúng trạng thái thị trường tại thời điểm chạy."], "knowledge_graph.png"),
    ("4.2.1. BỘ BA, THỰC THỂ VÀ QUAN HỆ", [
        "Một bộ ba có dạng (s, p, o), trong đó s là thực thể nguồn, p là quan hệ có tên và o là thực thể hoặc giá trị đích. Ví dụ: (MA20, MÔ_TẢ_XU_HƯỚNG_NGẮN_HẠN_CỦA, FPT.VN). Bộ ba này không khẳng định xu hướng sẽ tiếp diễn; nó chỉ diễn tả vai trò của chỉ báo.",
        "Thực thể được gắn type như Condition, Biomarker, Stock, TechnicalIndicator, Commodity hoặc MacroFactor. Thuộc tính domain tách ba không gian nghiệp vụ. Tên quan hệ dùng động từ viết hoa để người đọc có thể đọc thành câu.",
        "Việc chọn quan hệ là công đoạn có tính chủ quan. Báo cáo ưu tiên từ CÓ_THỂ_TÁC_ĐỘNG hoặc CÓ_QUAN_HỆ... thay vì GÂY_RA khi dữ liệu không đủ để chứng minh nhân quả."], None),
    ("4.2.2. KNOWLEDGE GRAPH CORE ENGINE", [
        "Lớp KnowledgeGraphCore bọc MultiDiGraph của NetworkX. add_entity quản lý nút và thuộc tính; add_relation thêm cạnh có hướng; neighbors trả về bằng chứng một bước; multi_hop tìm các đường đơn trong giới hạn số hop. Cấu trúc MultiDiGraph cho phép hai thực thể có nhiều loại quan hệ.",
        "Độ phức tạp của truy vấn lân cận tỷ lệ với bậc của nút. Truy vấn mọi đường đơn có thể tăng nhanh theo kích thước đồ thị, vì vậy max_hops phải được giới hạn. Với đồ thị sản xuất, cần chỉ mục, phân quyền và chiến lược truy vấn cụ thể thay vì duyệt toàn bộ.",
        "Toàn bộ cạnh được xuất thành knowledge_graph_triples.csv. Đây là điểm kiểm tra thuận tiện: mỗi dòng trong tệp phải đọc được như một phát biểu, không có nút mồ côi vô nghĩa và không chứa con số không rõ ngày."], "code_kg_core.png"),
    ("4.2.3. RDF VÀ LABELED PROPERTY GRAPH", [
        "RDF phù hợp khi cần URI toàn cục, liên kết dữ liệu và suy luận theo chuẩn W3C. Labeled Property Graph thuận tiện khi cạnh cần thuộc tính và truy vấn đường đi là trọng tâm. Bài chọn mô hình thứ hai vì mục tiêu là nguyên mẫu phân tích và trực quan hóa bằng Python.",
        "Nếu chuyển sang RDF, các entity_id cần ánh xạ thành IRI; relation thành predicate; giá trị số thành literal có datatype. Nếu chuyển sang Neo4j, type có thể thành node label, relation thành relationship type và các trường domain/source thành property.",
        "Không có mô hình nào tốt tuyệt đối. Tiêu chí chọn là khả năng tích hợp với hệ thống, yêu cầu suy luận, quy mô và kỹ năng vận hành của nhóm."], None),
    ("4.2.4. CHẤT LƯỢNG VÀ QUẢN TRỊ TRI THỨC", [
        "Một KG hữu ích cần hơn số lượng cạnh. Bốn kiểm tra tối thiểu gồm: tính duy nhất của định danh, tính hợp lệ của miền–phạm vi, nguồn gốc quan hệ và thời hạn của dữ liệu động. Các quan hệ y tế và tài chính phải có mức tin cậy hoặc câu chữ thể hiện độ bất định.",
        "Bài hiện lưu source ở lớp tài liệu và domain ở lớp đồ thị. Bản sản xuất nên bổ sung created_at, valid_from, valid_to, confidence và evidence_id cho từng cạnh. Khi đó, một quan hệ đã cũ có thể bị loại trước khi đưa vào prompt.",
        "Kiểm soát chất lượng cần vòng phản hồi của chuyên gia. Bác sĩ xác nhận ontology y tế; chuyên viên tài chính rà soát khái niệm và đơn vị; kỹ sư dữ liệu kiểm tra schema cùng pipeline cập nhật."], None),
    ("4.3. RAG — RETRIEVAL-AUGMENTED GENERATION", [
        "Pipeline RAG gồm ingestion, indexing, retrieval và generation. Văn bản được chia thành các đơn vị có nguồn; TF-IDF biến chúng thành vector; cosine similarity xếp hạng theo câu hỏi. Bước sinh ghép hai đoạn phù hợp nhất với các bộ ba liên quan.",
        "TF-IDF được chọn vì minh bạch, nhẹ và tái lập. Hạn chế là không hiểu đồng nghĩa sâu như embedding. Để giảm nhược điểm, câu hỏi được nhận diện miền bằng một từ điển alias và GraphRAG bổ sung các quan hệ từ đồ thị.",
        "Mỗi câu trả lời trả về cả documents, score, triples và domain. Cấu trúc này quan trọng hơn phần câu chữ: giao diện hoặc báo cáo có thể trình bày bằng chứng mà không phải phân tích ngược chuỗi văn bản."], "architecture.png"),
    ("4.3.1. NẠP, LẬP CHỈ MỤC VÀ TRUY XUẤT", [
        "Các tài liệu ngắn được xây từ mô tả bộ dữ liệu và kết quả thực nghiệm. Trong hệ thống lớn, ingestion cần đọc PDF, HTML hoặc cơ sở dữ liệu; chuẩn hóa Unicode; bỏ trùng; chia đoạn theo ngữ nghĩa; và gắn metadata trước khi embedding.",
        "TfidfVectorizer dùng unigram và bigram. Bigram giúp phân biệt các cụm như giá vàng, tiểu đường, lãi suất Fed. top_k=3 giữ ngữ cảnh ngắn và dễ kiểm tra. Precision@3 theo domain được dùng như phép đo đơn giản cho độ đúng của truy xuất.",
        "Điểm cosine chỉ phản ánh độ gần từ vựng, không phải độ đúng. Vì vậy báo cáo không diễn giải score như xác suất và không tự động loại bỏ nguồn chuyên môn chỉ vì score thấp."], "code_retriever.png"),
    ("4.3.2. TRUY XUẤT ĐỒ THỊ VÀ SUY LUẬN NHIỀU BƯỚC", [
        "graph_evidence nhận diện miền, chọn các nút cùng domain hoặc trùng từ khóa, rồi lấy quan hệ vào và ra. Phương án này đủ cho đồ thị nhỏ. Khi đồ thị lớn, cần entity linking, tìm kiếm theo embedding của nhãn và truy vấn Cypher có giới hạn.",
        "Suy luận nhiều bước được minh họa bởi chuỗi Rủi ro địa chính trị → Nhu cầu trú ẩn → Vàng quốc tế. Vector search có thể tìm một đoạn nói về địa chính trị nhưng không mặc nhiên biểu diễn hai quan hệ riêng. Đồ thị giúp giữ cấu trúc của lời giải thích.",
        "Đường đi chỉ là bằng chứng cấu trúc, không tự chứng minh nhân quả. Mọi kết luận vẫn phụ thuộc chất lượng cạnh và nguồn đã dùng để tạo cạnh."], "code_graphrag.png"),
    ("4.3.3. TỔNG HỢP CÂU TRẢ LỜI CÓ CĂN CỨ", [
        "Bộ sinh trong bài là deterministic synthesizer. Nó chọn hai tài liệu phù hợp, nối tối đa ba quan hệ đồ thị và thêm cảnh báo theo miền. Kết quả ít tự nhiên hơn LLM nhưng có lợi thế là chạy offline và không tự thêm sự kiện ngoài bằng chứng.",
        "Có thể thay bộ sinh bằng một LLM cục bộ hoặc dịch vụ API. Khi đó prompt phải tách rõ QUESTION, TEXT EVIDENCE, GRAPH EVIDENCE và yêu cầu từ chối nếu bằng chứng thiếu. Đầu ra nên ở dạng JSON để kiểm tra citation_id.",
        "Thiết kế production cần chống prompt injection từ tài liệu, lọc dữ liệu nhạy cảm, theo dõi phiên bản chỉ mục và ghi log các nguồn đã đưa cho mô hình."], None),
    ("4.3.4. ĐÁNH GIÁ RAG VÀ GRAPHRAG", [
        "Bộ kiểm thử gồm sáu câu hỏi, hai câu cho mỗi miền. vector_precision@3 đo tỷ lệ tài liệu thuộc đúng miền trong ba kết quả đầu. graph_evidence ghi nhận hệ thống có tìm được cạnh liên quan hay không. Điểm tổng hợp cộng một phần thưởng nhỏ khi cả hai kênh đều có bằng chứng.",
        f"Ở lần chạy này, precision@3 trung bình là {evaluation_df['vector_precision@3'].mean():.3f}; tỷ lệ câu hỏi có bằng chứng đồ thị là {evaluation_df['graph_evidence'].mean():.3f}; điểm bằng chứng lai là {evaluation_df['hybrid_evidence_score'].mean():.3f}.",
        "Đây là smoke test, chưa phải benchmark khoa học. Một đánh giá đầy đủ cần nhiều câu hỏi do người độc lập soạn, đáp án chuẩn, chấm faithfulness, answer relevance, citation correctness, độ trễ và chi phí."], "rag_benchmark.png"),

("4.4. KB VÀ RAG CHO BỆNH TIỂU ĐƯỜNG", [
    "Bộ dữ liệu diabetes của scikit-learn có 442 quan sát và 10 biến đầu vào đã chuẩn hóa. Biến đích là chỉ số định lượng mức tiến triển bệnh sau một năm. Dữ liệu không chứa tên, mã hồ sơ hay thông tin nhận dạng cá nhân.",
    "Bốn cấu hình được so sánh bằng K-fold 5 phần trên tập train: Linear Regression, Ridge, Random Forest và Gradient Boosting. Tập test 20% được giữ riêng cho đến khi đã chọn mô hình theo CV MAE.",
    f"Mô hình được chọn là {diabetes_metrics['Model']}; CV MAE = {diabetes_metrics['CV_MAE']:.2f}, test MAE = {diabetes_metrics['MAE']:.2f}, RMSE = {diabetes_metrics['RMSE']:.2f}, R² = {diabetes_metrics['R2']:.3f}. Tham số: {best_params}."], "kg_diabetes.png"),
    ("4.4.1. ONTOLOGY VÀ XỬ LÝ DỮ LIỆU Y TẾ", [
        "Ontology tối giản gồm Condition, Biomarker, RiskFactor, Lifestyle, Monitoring và Professional. Các quan hệ như LIÊN_QUAN_ĐẾN hoặc HỖ_TRỢ_QUẢN_LÝ được chọn để tránh biến một tương quan thành chỉ định điều trị.",
        "Dữ liệu không có tên bệnh nhân hay định danh cá nhân. Trong tình huống thực, hồ sơ y tế cần mã hóa, phân quyền theo vai trò, nhật ký truy cập và quy trình chấp thuận. Không nên đưa thông tin nhận dạng vào dịch vụ embedding công cộng khi chưa có cơ sở pháp lý.",
        f"Mô hình {diabetes_metrics['Model']} được chọn theo MAE trung bình của 5 fold. Permutation importance được tính trên tập test để cùng một cách giải thích áp dụng được cho cả mô hình tuyến tính và mô hình cây."], "code_diabetes_training.png"),
    ("4.4.2. THỰC NGHIỆM HỎI ĐÁP Y TẾ", [
        "Câu hỏi mẫu hỏi các yếu tố liên quan đến quản lý bệnh tiểu đường. Vector retriever tìm mô tả theo dõi đường huyết, HbA1c, huyết áp, cân nặng, ăn uống và vận động. Đồ thị bổ sung các cạnh nối bệnh với theo dõi và chuyên gia.",
        f"Câu trả lời sinh ra trong lần chạy: {demo_answers[0]['answer']}",
        "Biểu đồ đối chiếu giá trị thực - dự đoán và feature importance cho thấy mức độ phù hợp cũng như biến nào được mô hình sử dụng nhiều nhất."], "diabetes_model.png"),
    ("4.4.3. GIỚI HẠN VÀ HƯỚNG MỞ RỘNG Y TẾ", [
        "Bộ dữ liệu nhỏ, không đại diện cho dân số Việt Nam và không chứa bối cảnh điều trị. Chỉ số đích mang ý nghĩa nghiên cứu, khác với chẩn đoán đái tháo đường trong thực hành. Vì vậy không thể suy rộng độ chính xác mô hình thành hiệu quả lâm sàng.",
        "Hướng mở rộng là dùng dữ liệu đã khử định danh, xác định rõ biến kết cục, đánh giá theo nhóm tuổi/giới và mời chuyên gia lâm sàng kiểm tra ontology. Ngoài độ chính xác, cần đo calibration, fairness và khả năng giải thích.",
        "Với RAG y tế, mọi đoạn văn cần version và ngày hiệu lực. Hướng dẫn cũ phải được thu hồi khỏi chỉ mục thay vì chỉ thêm tài liệu mới."], None),

("4.5. KB VÀ RAG CHO CHỨNG KHOÁN", [
    f"Dữ liệu FPT.VN được lấy từ {market_sources['stock']}. Khoảng thời gian {stock_summary['start']} - {stock_summary['end']} có {stock_summary['observations']} quan sát. Giá đóng cửa gần nhất là {stock_summary['latest_close']:,.0f} {stock_summary['currency']}.",
    f"Lợi suất toàn kỳ là {stock_summary['period_return_pct']:.2f}%, biến động năm hóa {stock_summary['volatility_pct']:.2f}%, RSI14 {stock_summary['rsi14']:.1f}. Mỗi con số được gắn ngày chạy và nguồn dữ liệu.",
    "Knowledge Graph nối FPT.VN với giá gần nhất, MA20, MA60, RSI, thanh khoản, kết quả kinh doanh và tâm lý thị trường. Các quan hệ mô tả vai trò chỉ báo, không tự động tạo khuyến nghị mua bán."], "kg_stock.png"),
("4.5.1. DỮ LIỆU VÀ TIỀN XỬ LÝ FPT", [
    "Chuỗi OHLCV được sắp xếp tăng dần theo ngày; dòng thiếu Close bị loại. Return_1D, MA20, MA60, volatility20 và RSI14 được tính từ dữ liệu lịch sử. Đặc trưng bổ sung gồm thay đổi khối lượng, biên độ High-Low, chênh lệch Open-Close và các độ trễ một phiên.",
    f"Tại điểm cuối, MA20 = {stock_summary['ma20']:,.0f}, MA60 = {stock_summary['ma60']:,.0f}. Bộ học có {stock_model_info['train_size']} mẫu train và {stock_model_info['test_size']} mẫu test theo thứ tự thời gian.",
    "Biến đích là giá Close của phiên kế tiếp. Việc dịch target một bước rồi drop dòng cuối giúp đặc trưng tại ngày t không chứa giá của ngày t+1."], "fpt_market.png"),
("4.5.2. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH FPT", [
    "Ridge, Random Forest và Gradient Boosting được tuning bằng TimeSeriesSplit 5 phần. Khác K-fold ngẫu nhiên, mỗi fold chỉ huấn luyện trên quá khứ và xác nhận trên đoạn thời gian xảy ra sau đó. Tập test 20% cuối chuỗi không tham gia tuning.",
    f"Mô hình được chọn: {stock_model_info['Model']}; CV MAE = {stock_model_info['CV_MAE']:.2f}; test MAE = {stock_model_info['MAE']:.2f}; RMSE = {stock_model_info['RMSE']:.2f}; MAPE = {stock_model_info['MAPE']:.2f}%; R² = {stock_model_info['R2']:.3f}.",
    f"Độ đúng chiều biến động là {stock_model_info['Direction_Accuracy']:.1f}%. Persistence baseline có MAE {stock_model_info['Persistence_MAE']:.2f}; mức cải thiện của mô hình là {stock_model_info['Improvement_vs_Persistence_pct']:.2f}%. Báo cáo giữ nguyên kết quả kể cả khi baseline tốt hơn."], "stock_model.png"),
("4.5.3. HỎI ĐÁP CHỨNG KHOÁN VỚI GRAPHRAG", [
    f"Câu trả lời mẫu: {demo_answers[1]['answer']}",
    "Tài liệu cung cấp số liệu động; đồ thị giải thích vai trò của chỉ báo và liên kết multi-hop. Bộ lọc domain ngăn tài liệu vàng hoặc y tế đi vào câu trả lời chứng khoán.",
    "Kết quả mô hình được ghi thành tài liệu trong KB với model name, thời gian, MAE và R². Vì vậy RAG có thể dẫn lại đúng lần huấn luyện thay vì tạo ra độ chính xác không tồn tại."], "code_market_training.png"),
("4.5.4. RỦI RO VÀ HƯỚNG MỞ RỘNG", [
    "Corporate actions, thanh khoản, cách điều chỉnh giá và survivorship bias có thể làm sai phân tích. Yahoo Finance phù hợp bài tập nhưng nên đối chiếu nguồn chính thức nếu dùng trong nghiên cứu giao dịch.",
    "Dự báo Close phiên kế tiếp thường có R² cao do giá hôm nay gần giá ngày mai; vì vậy phải đọc cùng persistence baseline và directional accuracy. R² cao không đồng nghĩa chiến lược có lợi nhuận.",
    "Hướng mở rộng là walk-forward backtest có phí, trượt giá, khoảng tin cậy và kiểm tra thay đổi chế độ thị trường. Mọi đầu ra vẫn cần cảnh báo không phải khuyến nghị đầu tư."], None),

("4.6. KB VÀ RAG CHO GIÁ VÀNG", [
    f"Dữ liệu hợp đồng vàng GC=F được lấy từ {market_sources['gold']}. Khoảng {gold_summary['start']} - {gold_summary['end']} có {gold_summary['observations']} quan sát; giá gần nhất {gold_summary['latest_close']:,.2f} {gold_summary['currency']}/oz.",
    f"Lợi suất toàn kỳ {gold_summary['period_return_pct']:.2f}%, biến động năm hóa {gold_summary['volatility_pct']:.2f}%, RSI14 {gold_summary['rsi14']:.1f}. GC=F là tham chiếu quốc tế, không đồng nhất với giá vàng miếng SJC.",
    "Knowledge Graph mô tả lãi suất Fed, USD, lạm phát, địa chính trị và nhu cầu trú ẩn. Những cạnh này là khung giải thích có điều kiện, không phải quan hệ nhân quả cố định."], "kg_gold.png"),
("4.6.1. DỮ LIỆU, CHỈ BÁO VÀ KNOWLEDGE GRAPH VÀNG", [
    f"MA20 hiện là {gold_summary['ma20']:,.2f}, MA60 là {gold_summary['ma60']:,.2f}. Chuỗi giữ nguyên lịch phiên của nguồn; không nội suy ngày nghỉ. Biến đích là Close phiên kế tiếp.",
    "Đặc trưng dùng cùng một công thức với FPT để so sánh công bằng: giá và độ trễ, return, MA20, MA60, volatility20, RSI14, thay đổi volume, biên độ ngày và chênh lệch Open-Close.",
    "Chuỗi Rủi ro địa chính trị -> Nhu cầu trú ẩn -> Vàng quốc tế minh họa truy vấn nhiều bước. Quan hệ Vàng quốc tế -> Giá vàng SJC chỉ là tham chiếu, cần thêm tỷ giá và cơ chế thị trường trong nước để giải thích chênh lệch."], "gold_market.png"),
("4.6.2. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH VÀNG", [
    "Ridge, Random Forest và Gradient Boosting được tuning bằng TimeSeriesSplit 5 phần trên 80% đầu chuỗi. 20% cuối được dùng duy nhất cho kiểm thử ngoài mẫu. Tiêu chí lựa chọn là CV MAE.",
    f"Mô hình được chọn: {gold_model_info['Model']}; CV MAE = {gold_model_info['CV_MAE']:.3f}; test MAE = {gold_model_info['MAE']:.3f}; RMSE = {gold_model_info['RMSE']:.3f}; MAPE = {gold_model_info['MAPE']:.2f}%; R² = {gold_model_info['R2']:.3f}.",
    f"Độ đúng chiều là {gold_model_info['Direction_Accuracy']:.1f}%. Persistence baseline có MAE {gold_model_info['Persistence_MAE']:.3f}; mức cải thiện là {gold_model_info['Improvement_vs_Persistence_pct']:.2f}%. Các chỉ số được tính trực tiếp từ phần test theo thời gian."], "gold_model.png"),
("4.6.3. HỎI ĐÁP VĨ MÔ VỚI GRAPHRAG", [
    f"Câu trả lời mẫu: {demo_answers[2]['answer']}",
    "Truy xuất văn bản nêu điều kiện và độ bất định; đồ thị cung cấp các cạnh thường gặp và đường đi qua nhu cầu trú ẩn. Hai nguồn bằng chứng cùng xuất hiện giúp câu trả lời tránh tuyệt đối hóa.",
    "Nếu thay bộ sinh mẫu bằng LLM, prompt phải cấm tự tạo mục tiêu giá hoặc xác suất tăng giảm khi evidence không cung cấp. Citation ID và timestamp nên là trường bắt buộc trong JSON đầu ra."], "code_graphrag.png"),
("4.6.4. GIỚI HẠN VÀ HƯỚNG MỞ RỘNG", [
    "Hợp đồng tương lai có ngày đáo hạn và cơ chế nối chuỗi. Spot XAU/USD, futures GC=F và vàng SJC là ba sản phẩm khác nhau; không thể trộn USD/oz với VND/lượng nếu chưa quy đổi.",
    "Mô hình hiện chưa dùng lãi suất thực, DXY, tỷ giá USD/VND hay vị thế ETF. Hướng mở rộng là gắn timestamp cho các yếu tố vĩ mô, thử độ trễ và lưu confidence cùng evidence trên mỗi cạnh.",
    "Tương tự chứng khoán, dự báo một bước chỉ minh họa quy trình machine learning có kiểm soát. Nó không phải khuyến nghị mua, bán hay nắm giữ vàng."], None),
    ("4.7. DEPLOYMENT", [
        "Notebook sinh một ứng dụng web tương tác trong vercel_app. Người dùng có thể nhập biến tiểu đường, chỉ báo FPT hoặc chỉ báo vàng rồi nhận dự đoán ngay trên trình duyệt. Kết quả không phải số viết sẵn: notebook xuất mean, scale, coefficient và intercept từ đúng pipeline đã huấn luyện sang JavaScript.",
        "Cùng trang web có Chatbot Sức khỏe & Dinh dưỡng Bệnh Tiểu đường. Bộ truy xuất chấm điểm từ khóa, chọn đoạn phù hợp trong kho tri thức và đưa cả nguồn lẫn quan hệ Knowledge Graph vào câu trả lời. Nội dung dựa trên WHO, CDC và ADA; chatbot không chẩn đoán hoặc thay đổi liều thuốc.",
        f"Ứng dụng đã triển khai tại <link href='{DEPLOY_URL}' color='black'>{DEPLOY_URL}</link>. Mã nguồn và notebook đặt tại <link href='{GITHUB_URL}' color='black'>{GITHUB_URL}</link>."], "web_app_preview.png"),
    ("4.7.1. SUY LUẬN MÔ HÌNH TRÊN TRÌNH DUYỆT", [
        "Phần compute nặng gồm tải dữ liệu, tạo đặc trưng, cross-validation và fit mô hình chạy trong notebook. Phần web chỉ nhận tham số cuối cùng của StandardScaler và Ridge. Với từng mẫu x, trình duyệt tính z=(x-mean)/scale, sau đó y=intercept+sum(coef*z). Vì vậy cùng đầu vào sẽ cho cùng kết quả như pipeline Python.",
        "Biểu mẫu tiểu đường dùng đủ 10 thuộc tính của bộ dữ liệu scikit-learn. Hai biểu mẫu thị trường hiển thị sáu biến dễ diễn giải; năm biến còn lại lấy từ quan sát mới nhất nhưng vẫn tham gia phép dự đoán. Ngày dữ liệu và đơn vị được hiển thị ngay cạnh biểu mẫu.",
        "Kiến trúc tĩnh phù hợp Vercel: không cần lưu API key, phản hồi tức thời và không gửi dữ liệu biểu mẫu về máy chủ. Khi cập nhật dữ liệu, chỉ cần Run All notebook và deploy lại index.html."], "code_deployment.png"),
    ("4.7.2. CHATBOT RAG SỨC KHỎE VÀ DINH DƯỠNG", [
        "Kho tri thức chatbot bao gồm phương pháp chiếc đĩa, carbohydrate, vận động, theo dõi đường huyết và xử trí tình huống khẩn cấp. Câu hỏi tiếng Việt được chuẩn hóa dấu, so khớp với cụm từ khóa có trọng số, rồi lấy tài liệu đạt điểm cao nhất.",
        "Mỗi phản hồi hiển thị tên nguồn và một đường bằng chứng dạng subject - relation - object. Cách trình bày này giúp người đọc phân biệt nội dung tài liệu với cấu trúc KG. Nếu câu hỏi chứa dấu hiệu như bất tỉnh, co giật hoặc khó thở, hệ thống ưu tiên hướng dẫn tìm trợ giúp khẩn cấp.",
        "Chatbot là công cụ giáo dục sức khỏe. Nó không thu thập bệnh án, không chẩn đoán, không kê đơn và không thay thế bác sĩ. Những câu hỏi cá nhân về mục tiêu đường huyết hoặc liều thuốc được hướng về nhân viên y tế."], None),
    ("4.7.3. KIỂM THỬ TRIỂN KHAI", [
        "Kiểm thử gồm ba lớp: so sánh phép tính JavaScript với pipeline Python, smoke test ba nút dự đoán, và thử các nhóm câu hỏi ăn uống, vận động, theo dõi cùng tình huống khẩn cấp. Liên kết nguồn phải mở đúng trang WHO, CDC hoặc ADA.",
        "Trước khi nộp, PDF được render thành ảnh để kiểm tra font, mục lục, số trang và độ nét. Web được kiểm tra trên bố cục desktop và responsive, đặc biệt là khả năng nhập số, chuyển tab và gửi câu hỏi bằng phím Enter.",
        "Deployment hoàn thành khi URL production truy cập được, ba biểu mẫu trả kết quả, chatbot trả bằng chứng và số liệu mô hình khớp với lần chạy notebook tạo báo cáo."], None),
    ("ĐÁNH GIÁ TỔNG HỢP", [
        "Kết quả cho thấy một notebook có thể bao quát tải dữ liệu, tiền xử lý, KG, RAG, đánh giá, báo cáo và web. GraphRAG lai cung cấp thêm loại bằng chứng mà Vector RAG không có: quan hệ có tên và đường đi giữa các thực thể.",
        "Đổi lại, KG yêu cầu thiết kế ontology, kiểm soát quan hệ và cập nhật theo thời gian. Chi phí này chỉ đáng bỏ ra khi bài toán cần giải thích, liên kết đa bước hoặc quản trị tri thức dài hạn.",
        "Bài làm đạt mục tiêu nguyên mẫu tái lập. Hạn chế chính là corpus nhỏ, đánh giá ít câu hỏi, generator dạng mẫu và chưa có chuyên gia miền thẩm định. Những giới hạn này được ghi rõ thay vì suy rộng kết quả."], "rag_benchmark.png"),
    ("KẾT LUẬN", [
        "Knowledge Graph và RAG bổ trợ nhau theo cách thực dụng: văn bản đem lại ngữ cảnh linh hoạt, đồ thị đem lại cấu trúc và provenance. Trên ba miền thử nghiệm, cùng một pipeline vẫn giữ được nguồn, thời điểm và cảnh báo riêng.",
        "Điều quan trọng nhất rút ra là không nên coi GraphRAG như một nhãn công nghệ thay thế cho thiết kế dữ liệu. Chất lượng câu trả lời bắt đầu từ việc định nghĩa thực thể, quan hệ, nguồn và phạm vi kết luận.",
        "Notebook kèm theo là sản phẩm có thể chạy lại. PDF và dashboard chỉ là hai cách trình bày cùng một trạng thái kết quả, nhờ đó giảm sai lệch giữa code và báo cáo."], None),
]

for title, paragraphs, figure in pages:
    story.append(P(title, "VNChapter" if title.count(".") <= 1 else "VNHeading"))
    for paragraph in paragraphs:
        story.append(P(paragraph))
    if figure:
        img_path = FIG_DIR / figure
        if img_path.exists():
            img = report_image(img_path)
            story += [Spacer(1, .15*cm), img, P(f"Hình: {title}", "VNCaption")]
    story.append(PageBreak())


# Bảng kết quả chi tiết
story += [P("PHỤ LỤC A - TỔNG HỢP HUẤN LUYỆN BA MIỀN", "VNChapter")]
training_rows = [["Miền", "Mô hình được chọn", "CV MAE", "Test MAE", "Test R²"]] + [
    ["Tiểu đường", diabetes_metrics["Model"], f"{diabetes_metrics['CV_MAE']:.3f}",
     f"{diabetes_metrics['MAE']:.3f}", f"{diabetes_metrics['R2']:.3f}"],
    ["Chứng khoán FPT", stock_model_info["Model"], f"{stock_model_info['CV_MAE']:.3f}",
     f"{stock_model_info['MAE']:.3f}", f"{stock_model_info['R2']:.3f}"],
    ["Giá vàng GC=F", gold_model_info["Model"], f"{gold_model_info['CV_MAE']:.3f}",
     f"{gold_model_info['MAE']:.3f}", f"{gold_model_info['R2']:.3f}"],
]
story += [report_table(training_rows, [3.2*cm, 4.0*cm, 2.7*cm, 2.7*cm, 2.7*cm]),
          Spacer(1, .45*cm),
          P("Mô hình được chọn theo CV MAE trên tập train. Chỉ số test được tính đúng một lần trên phần dữ liệu giữ lại."),
          PageBreak()]

story += [P("PHỤ LỤC B - KẾT QUẢ TRUY XUẤT", "VNChapter")]

eval_data = [["Câu hỏi", "Miền", "P@3", "Graph", "Hybrid"]] + [
    [P(r["query"], "VNSmall"), r["expected_domain"], f"{r['vector_precision@3']:.2f}", str(r["graph_evidence"]), f"{r['hybrid_evidence_score']:.2f}"]
    for _, r in evaluation_df.iterrows()
]
story += [report_table(eval_data, [8.4*cm, 2.0*cm, 1.6*cm, 1.6*cm, 1.8*cm], font_size=7.5),
          Spacer(1, .5*cm), P("Bảng này là phép kiểm tra hồi quy cho pipeline. Nó không thay thế đánh giá câu trả lời bởi chuyên gia."), PageBreak()]

story += [P("PHỤ LỤC C - MẪU BỘ BA TRI THỨC", "VNChapter")]
triple_data = [["Subject", "Predicate", "Object"]] + [
    [P(str(r.Subject), "VNSmall"), P(str(r.Predicate), "VNSmall"), P(str(r.Object), "VNSmall")]
    for r in triples_df.head(20).itertuples()
]
story += [report_table(triple_data, [4.9*cm, 5.2*cm, 5.4*cm], font_size=7.2), PageBreak()]

references = [
    "[1] W3C, RDF 1.1 Concepts and Abstract Syntax, W3C Recommendation, 2014.",
    "[2] T. Berners-Lee, J. Hendler, O. Lassila, The Semantic Web, Scientific American, 2001.",
    "[3] K. Guu et al., REALM: Retrieval-Augmented Language Model Pre-Training, ICML, 2020.",
    "[4] P. Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks, NeurIPS, 2020.",
    "[5] D. Edge et al., From Local to Global: A Graph RAG Approach to Query-Focused Summarization, 2024.",
    "[6] scikit-learn developers, Diabetes dataset and TfidfVectorizer documentation.",
    "[7] NetworkX developers, MultiDiGraph documentation.",
    "[8] Yahoo Finance chart endpoint, mã FPT.VN và GC=F; truy cập tại thời điểm chạy notebook.",
    "[9] World Health Organization, Diabetes fact sheet, https://www.who.int/news-room/fact-sheets/detail/diabetes.",
    "[10] Centers for Disease Control and Prevention, Diabetes Meal Planning, https://www.cdc.gov/diabetes/healthy-eating/diabetes-meal-planning.html.",
    "[11] American Diabetes Association, Eating Healthy, https://diabetes.org/food-nutrition/eating-healthy.",
    "[12] Vercel Documentation, Static Deployments and CLI.",
    f"[13] Mã nguồn bài làm: <link href='{GITHUB_URL}' color='black'>{GITHUB_URL}</link>.",
    f"[14] Web triển khai: <link href='{DEPLOY_URL}' color='black'>{DEPLOY_URL}</link>.",
]
story += [P("TÀI LIỆU THAM KHẢO", "VNChapter")]
story += [P(ref) for ref in references]
story += [Spacer(1, .5*cm), P(f"Báo cáo được sinh tự động từ notebook lúc {RUN_AT:%d/%m/%Y %H:%M:%S %Z}. Dữ liệu nguồn: FPT — {market_sources['stock']}; vàng — {market_sources['gold']}.", "VNSmall")]

doc = ReportDoc(str(REPORT_PATH))
doc.multiBuild(story)
print("Đã tạo báo cáo:", REPORT_PATH)

Đã tạo báo cáo: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/Tieuluan02_06_Nhom2_Giangkt.pdf


In [10]:
# 6. KIỂM TRA ĐẦU RA VÀ TRIỂN KHAI TÙY CHỌN
assert REPORT_PATH.exists() and REPORT_PATH.stat().st_size > 100_000
assert (WEB_DIR / "index.html").exists()
assert (DATA_DIR / "knowledge_graph_triples.csv").exists()

try:
    from pypdf import PdfReader
except Exception:
    bundled_site = Path("/home/jellalaz/.cache/codex-runtimes/codex-primary-runtime/dependencies/python")
    if bundled_site.exists() and str(bundled_site) not in sys.path:
        sys.path.append(str(bundled_site))
    from pypdf import PdfReader
page_count = len(PdfReader(str(REPORT_PATH)).pages)
print(f"PDF hợp lệ: {page_count} trang, {REPORT_PATH.stat().st_size/1024/1024:.2f} MB")
print("Web package:", WEB_DIR)

# Để deploy thực sự: đặt biến môi trường VERCEL_TOKEN và chuyển cờ thành True.
# Token tuyệt đối không được ghi trực tiếp vào notebook.
DEPLOY_TO_VERCEL = False
if DEPLOY_TO_VERCEL:
    token = os.getenv("VERCEL_TOKEN")
    if not token:
        raise RuntimeError("Thiếu biến môi trường VERCEL_TOKEN")
    subprocess.run(
        ["npx", "vercel", "--prod", "--yes", "--token", token],
        cwd=WEB_DIR, check=True
    )
else:
    print("Production URL cấu hình trong báo cáo:", DEPLOY_URL)

PDF hợp lệ: 42 trang, 3.70 MB
Web package: /home/jellalaz/Documents/Jellalaz/DATA_CODE/PYTHON/RAG/vercel_app
Production URL cấu hình trong báo cáo: https://rag-httm-ai.vercel.app
